# EBSD Map Stitching with Grain Graphs and Percolation Logic

**Right + Down Tile-Pair Variant:** this notebook evaluates both horizontal right-neighbor pairs and vertical down-neighbor pairs.

This notebook demonstrates an EBSD stitching workflow built around MSE 521-style ideas: orientations as rotations in SO(3), misorientation, grain-boundary graphs, ODF/pole-figure-inspired orientation fingerprints, and percolation-style connectivity.

The notebook can load a real `.ang` file, but it also generates synthetic EBSD-like data so the full workflow is runnable without paid software or proprietary files. The core approach is:

1. Parse or synthesize an EBSD map.
2. Correct scan-direction distortion from sample tilt.
3. Create overlapping artificial tiles and apply a known distortion to one tile.
4. Segment approximate grains.
5. Build grain-boundary graphs.
6. Compare graph/percolation fingerprints near the overlap.
7. Match grains across tile boundaries with GATv2 when available, then estimate an affine transform with RANSAC.
8. Stitch and validate.

Assumptions and TODOs:

- Euler angles are assumed to be Bunge ZXZ angles in radians unless the parser detects degree-like values.
- Misorientation uses configurable crystal symmetry operators. Phase 1 defaults to cubic; update `PHASE_SYMMETRY` for multiphase datasets.
- Grain segmentation is approximate when no grain IDs are present. It uses 4-neighbor connected components under a misorientation threshold.
- The matching pipeline uses a GATv2 graph neural network when `torch` and `torch_geometric` are installed, with transparent graph/percolation scoring as a fallback.



## 1. Imports and Setup

This cell imports required packages, detects optional EBSD/GNN packages, and sets the random seed. The notebook only requires common scientific Python packages.

In [1]:
!pip install -q torch-geometric
import importlib.util
import math
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from scipy import ndimage
from scipy.spatial import cKDTree
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from skimage.measure import label, regionprops
from skimage.transform import AffineTransform
from skimage.measure import ransac

SEED = 521
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

OPTIONAL = {name: importlib.util.find_spec(name) is not None for name in [
    'orix', 'kikuchipy', 'torch', 'torch_geometric'
]}
OPTIONAL


# Colab / H100 setup ---------------------------------------------------------
# In Colab: Runtime -> Change runtime type -> GPU -> H100, then run:
#   from google.colab import drive
#   drive.mount('/content/drive')
# and set COLAB_PROJECT_ROOT to your Drive project folder if needed.
try:
    import torch
    TORCH_AVAILABLE = True
except Exception:
    torch = None
    TORCH_AVAILABLE = False

DEVICE = torch.device('cuda' if TORCH_AVAILABLE and torch.cuda.is_available() else 'cpu') if TORCH_AVAILABLE else None
if TORCH_AVAILABLE:
    print('Torch device:', DEVICE)
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
else:
    print('Torch not available yet. Install torch/torch-geometric before the GATv2 sections.')

COLAB_PROJECT_ROOT = Path('/content/drive/MyDrive/ebsd_stitching') if Path('/content').exists() else Path('.')
USE_COLAB_DRIVE_PATHS = False


^C


ModuleNotFoundError: No module named 'networkx'

## Colab H100 Notes

For Colab, choose **Runtime > Change runtime type > GPU > H100** before running the notebook. If your Colab runtime does not already include PyTorch Geometric, run the install cell below once, then restart the runtime.

If your data is on Google Drive, mount Drive and set `USE_COLAB_DRIVE_PATHS = True` plus `COLAB_PROJECT_ROOT` to the folder that contains `data/file1_cropped` and `data/file2_cropped`.

In [ ]:
# Optional Colab setup cell. Leave commented until you need it.
# !pip install -q torch-geometric
# from google.colab import drive
# drive.mount('/content/drive')
# USE_COLAB_DRIVE_PATHS = True
# COLAB_PROJECT_ROOT = Path('/content/drive/MyDrive/ebsd_stitching')


## 2. Load EBSD `.ang` File

Set `ANG_PATH` to a real `.ang` file. If it is `None` or the file cannot be read, synthetic EBSD-like data are generated. The parser is flexible: it skips header/comment lines, detects numeric rows, and maps common TSL/OIM columns to a standard dataframe.

Expected output columns are `phi1`, `Phi`, `phi2`, `x`, `y`, `IQ`, `CI`, and `phase`. Extra columns are retained with generic names.

In [ ]:
# Provide your own file path to use a real EBSD map.
# Keep ANG_PATH = None for the fully runnable synthetic demonstration.
ANG_PATH = None

# Local files are listed for convenience, but not auto-loaded because real ANG maps can be large.
candidate_paths = sorted(Path('.').glob('**/*.ang'))
print('Local .ang candidates:', [str(p) for p in candidate_paths[:5]])
print('Current ANG_PATH:', ANG_PATH)


In [ ]:
def _configure_optional_reader_cache():
    """Keep optional package cache/config writes local to the project."""
    import os
    cache_root = Path('.cache')
    os.environ.setdefault('MPLCONFIGDIR', str(cache_root / 'matplotlib'))
    os.environ.setdefault('XDG_CACHE_HOME', str(cache_root))
    os.environ.setdefault('NUMBA_CACHE_DIR', str(cache_root / 'numba'))


def _first_matching_prop(xmap, aliases, default):
    for name in aliases:
        if name in xmap.prop:
            values = np.asarray(xmap.prop[name])
            if values.ndim > 1:
                values = values[:, 0]
            return values, name
    return np.full(xmap.size, default), None


def _parse_ang_with_orix(path, max_rows=None):
    """Load an ANG orientation map with orix, installed with kikuchipy."""
    _configure_optional_reader_cache()
    import orix.io

    xmap = orix.io.load(path)
    eulers = xmap.rotations.to_euler()
    iq, iq_source = _first_matching_prop(xmap, ['iq', 'imagequality', 'ind'], 1.0)
    ci, ci_source = _first_matching_prop(xmap, ['ci', 'confidenceindex', 'dp', 'rel'], 1.0)
    sem_signal, sem_source = _first_matching_prop(xmap, ['detector_signal', 'ds'], 0.0)
    fit, fit_source = _first_matching_prop(xmap, ['fit', 'patternfit'], 0.0)

    df = pd.DataFrame({
        'phi1': eulers[:, 0], 'Phi': eulers[:, 1], 'phi2': eulers[:, 2],
        'x': xmap.x, 'y': xmap.y, 'IQ': iq, 'CI': ci,
        'phase': xmap.phase_id.astype(int), 'SEM_signal': sem_signal, 'fit': fit,
    })
    used_props = {name for name in [iq_source, ci_source, sem_source, fit_source] if name}
    for name, values in xmap.prop.items():
        if name in used_props:
            continue
        arr = np.asarray(values)
        if arr.ndim == 1:
            df[name] = arr

    if max_rows is not None:
        df = df.iloc[:max_rows].copy()
    df.attrs['source_path'] = str(path)
    df.attrs['angles_detected_as'] = 'radians'
    df.attrs['loader'] = 'orix'
    return df


def parse_ang(path, max_rows=None):
    """Load a TSL/OIM-style .ang file, preferring orix when available."""
    path = Path(path)
    try:
        return _parse_ang_with_orix(path, max_rows=max_rows)
    except Exception as exc:
        print(f'orix ANG reader unavailable or failed ({exc}); using fallback parser')

    numeric_rows = []
    header_lines = []
    with path.open('r', errors='ignore') as f:
        for line in f:
            stripped = line.strip()
            if not stripped:
                continue
            parts = stripped.split()
            try:
                row = [float(p) for p in parts]
            except ValueError:
                header_lines.append(stripped)
                continue
            numeric_rows.append(row)
            if max_rows is not None and len(numeric_rows) >= max_rows:
                break

    if not numeric_rows:
        raise ValueError(f'No numeric EBSD rows found in {path}')

    ncols = max(len(r) for r in numeric_rows)
    arr = np.full((len(numeric_rows), ncols), np.nan, dtype=float)
    for i, r in enumerate(numeric_rows):
        arr[i, :len(r)] = r

    common_cols = ['phi1', 'Phi', 'phi2', 'x', 'y', 'IQ', 'CI', 'phase', 'SEM_signal', 'fit']
    cols = common_cols[:ncols] + [f'extra_{i}' for i in range(max(0, ncols - len(common_cols)))]
    df = pd.DataFrame(arr, columns=cols[:ncols])

    # Many ANG files store Euler angles in radians. If they look like degrees, convert to radians.
    angle_max = df[['phi1', 'Phi', 'phi2']].to_numpy().max()
    df.attrs['angles_detected_as'] = 'degrees' if angle_max > 2 * np.pi + 1e-3 else 'radians'
    if df.attrs['angles_detected_as'] == 'degrees':
        df[['phi1', 'Phi', 'phi2']] = np.deg2rad(df[['phi1', 'Phi', 'phi2']])

    for col, default in [('IQ', 1.0), ('CI', 1.0), ('phase', 1)]:
        if col not in df:
            df[col] = default

    df['phase'] = df['phase'].fillna(0).astype(int)
    df.attrs['source_path'] = str(path)
    df.attrs['header_preview'] = header_lines[:20]
    return df


def generate_synthetic_ebsd(nx=120, ny=90, n_grains=45, noise_deg=0.7):
    """Generate EBSD-like grid data using Voronoi grains with smooth quality fields."""
    xs, ys = np.meshgrid(np.arange(nx, dtype=float), np.arange(ny, dtype=float))
    pts = np.column_stack([xs.ravel(), ys.ravel()])
    centers = np.column_stack([rng.uniform(0, nx, n_grains), rng.uniform(0, ny, n_grains)])
    grain_id = cKDTree(centers).query(pts)[1]

    base_eulers = np.column_stack([
        rng.uniform(0, 2*np.pi, n_grains),
        rng.uniform(0, np.pi, n_grains),
        rng.uniform(0, 2*np.pi, n_grains),
    ])
    noise = np.deg2rad(noise_deg) * rng.normal(size=(pts.shape[0], 3))
    eulers = base_eulers[grain_id] + noise
    eulers[:, 0] %= 2*np.pi
    eulers[:, 1] = np.clip(eulers[:, 1], 0, np.pi)
    eulers[:, 2] %= 2*np.pi

    dist_to_center = np.linalg.norm(pts - centers[grain_id], axis=1)
    IQ = 100 + 25*np.exp(-dist_to_center / 25) + rng.normal(0, 4, pts.shape[0])
    CI = np.clip(0.7 + 0.2*rng.random(pts.shape[0]) - 0.002*dist_to_center, 0, 1)

    df = pd.DataFrame({
        'phi1': eulers[:, 0], 'Phi': eulers[:, 1], 'phi2': eulers[:, 2],
        'x': pts[:, 0], 'y': pts[:, 1], 'IQ': IQ, 'CI': CI, 'phase': 1,
        'grain_id_true': grain_id.astype(int),
    })
    df.attrs['source_path'] = 'synthetic'
    df.attrs['angles_detected_as'] = 'radians'
    return df


try:
    ebsd = parse_ang(ANG_PATH) if ANG_PATH else generate_synthetic_ebsd()
except Exception as exc:
    print(f'Falling back to synthetic data because loading failed: {exc}')
    ebsd = generate_synthetic_ebsd()

print('Source:', ebsd.attrs.get('source_path'))
print('Detected angle units:', ebsd.attrs.get('angles_detected_as'))
print('Rows:', len(ebsd), 'Columns:', list(ebsd.columns))
display(ebsd.head())


## 3. Orientation Math in SO(3)

The EBSD orientation at each point is represented by a rotation matrix `g` in SO(3). For Bunge Euler angles `(phi1, Phi, phi2)`, the convention used here is `g = Rz(phi1) Rx(Phi) Rz(phi2)`. Misorientation is the angle of `delta_g = g1 @ g2.T`:

`theta = arccos((trace(delta_g) - 1) / 2)`

TODO: replace this simple SO(3) angle with the minimum angle over crystal and sample symmetry operators.

In [ ]:
def euler_to_matrix(phi1, Phi, phi2):
    """Bunge Euler angles to one 3x3 rotation matrix using Rz(phi1) Rx(Phi) Rz(phi2)."""
    c1, s1 = np.cos(phi1), np.sin(phi1)
    c, s = np.cos(Phi), np.sin(Phi)
    c2, s2 = np.cos(phi2), np.sin(phi2)
    return np.array([
        [c1*c2 - s1*s2*c,  s1*c2 + c1*s2*c,  s2*s],
        [-c1*s2 - s1*c2*c, -s1*s2 + c1*c2*c, c2*s],
        [s1*s,              -c1*s,              c],
    ])


def euler_to_matrices(eulers):
    return np.stack([euler_to_matrix(a, b, c) for a, b, c in np.asarray(eulers)])


DEFAULT_CRYSTAL_SYMMETRY = 'cubic'
PHASE_SYMMETRY = {
    0: 'identity',
    1: 'cubic',
}


def _proper_cubic_symmetry_operators():
    """Return the 24 proper rotation symmetries of the cubic crystal class."""
    from itertools import permutations, product

    ops = []
    for perm in permutations(range(3)):
        P = np.zeros((3, 3), dtype=float)
        for row, col in enumerate(perm):
            P[row, col] = 1.0
        for signs in product([-1.0, 1.0], repeat=3):
            S = np.diag(signs) @ P
            if np.linalg.det(S) > 0.5:
                ops.append(S)
    return np.stack(ops)


SYMMETRY_OPERATORS = {
    'identity': np.eye(3)[None, :, :],
    'cubic': _proper_cubic_symmetry_operators(),
}


def symmetry_name_for_phase(phase):
    """Map ANG/orix phase IDs to crystal symmetry names.

    Update PHASE_SYMMETRY for multiphase datasets. Alloy 718 and most FCC/BCC
    engineering alloy examples are cubic, so phase 1 defaults to cubic.
    """
    if phase is None:
        return DEFAULT_CRYSTAL_SYMMETRY
    try:
        if pd.isna(phase):
            return DEFAULT_CRYSTAL_SYMMETRY
        return PHASE_SYMMETRY.get(int(round(float(phase))), DEFAULT_CRYSTAL_SYMMETRY)
    except Exception:
        return DEFAULT_CRYSTAL_SYMMETRY


def phase_id(value, default=1):
    """Robust integer phase ID conversion for ANG/orix values."""
    try:
        if value is None or pd.isna(value):
            return default
        return int(round(float(value)))
    except Exception:
        return default


def misorientation_deg(g1, g2, phase_a=None, phase_b=None, symmetry=None):
    """Minimum disorientation angle in degrees, with crystal symmetry when known.

    This evaluates S_i g1 g2^T S_j^T over the crystal symmetry group.
    Using both sides matters for production EBSD because equivalent crystal
    orientations should collapse to the same physical disorientation.
    """
    pa = phase_id(phase_a, default=None)
    pb = phase_id(phase_b, default=None)
    if pa is not None and pb is not None and pa != pb:
        return 180.0
    sym_name = symmetry or symmetry_name_for_phase(pa if pa is not None else pb)
    ops = SYMMETRY_OPERATORS.get(sym_name, SYMMETRY_OPERATORS['identity'])
    delta = g1 @ g2.T
    best = 180.0
    for S_left in ops:
        left_delta = S_left @ delta
        for S_right in ops:
            sym_delta = left_delta @ S_right.T
            cos_theta = (np.trace(sym_delta) - 1.0) / 2.0
            theta = np.rad2deg(np.arccos(np.clip(cos_theta, -1.0, 1.0)))
            if theta < best:
                best = float(theta)
    return best

def texture_bin_from_euler(phi1, Phi, phi2):
    """Lightweight texture bucket for per-texture validation reporting."""
    phi_deg = float(np.rad2deg(Phi) % 180.0)
    if phi_deg < 35.0:
        phi_band = 'low_Phi'
    elif phi_deg < 70.0:
        phi_band = 'mid_Phi'
    else:
        phi_band = 'high_Phi'
    phi1_sector = int((float(np.rad2deg(phi1) % 360.0)) // 90.0)
    return f'{phi_band}_phi1Q{phi1_sector}'


def texture_bin_from_row(row):
    return texture_bin_from_euler(row.phi1, row.Phi, row.phi2)


def matrix_to_quaternion(R):
    """Convert a rotation matrix to a normalized quaternion [w, x, y, z]."""
    tr = np.trace(R)
    if tr > 0:
        S = math.sqrt(tr + 1.0) * 2
        q = np.array([0.25*S, (R[2,1]-R[1,2])/S, (R[0,2]-R[2,0])/S, (R[1,0]-R[0,1])/S])
    else:
        i = int(np.argmax(np.diag(R)))
        if i == 0:
            S = math.sqrt(1.0 + R[0,0] - R[1,1] - R[2,2]) * 2
            q = np.array([(R[2,1]-R[1,2])/S, 0.25*S, (R[0,1]+R[1,0])/S, (R[0,2]+R[2,0])/S])
        elif i == 1:
            S = math.sqrt(1.0 + R[1,1] - R[0,0] - R[2,2]) * 2
            q = np.array([(R[0,2]-R[2,0])/S, (R[0,1]+R[1,0])/S, 0.25*S, (R[1,2]+R[2,1])/S])
        else:
            S = math.sqrt(1.0 + R[2,2] - R[0,0] - R[1,1]) * 2
            q = np.array([(R[1,0]-R[0,1])/S, (R[0,2]+R[2,0])/S, (R[1,2]+R[2,1])/S, 0.25*S])
    q = q / np.linalg.norm(q)
    return q if q[0] >= 0 else -q


test_g = euler_to_matrix(ebsd.loc[0, 'phi1'], ebsd.loc[0, 'Phi'], ebsd.loc[0, 'phi2'])
print('det(g)=', np.linalg.det(test_g), 'orthogonality error=', np.linalg.norm(test_g @ test_g.T - np.eye(3)))
print('Default crystal symmetry:', DEFAULT_CRYSTAL_SYMMETRY, 'operators=', len(SYMMETRY_OPERATORS[DEFAULT_CRYSTAL_SYMMETRY]))



## 4. EBSD Vertical Distortion Correction

A common geometric correction for EBSD scan coordinates accounts for the sample tilt angle. If `y_scan` is the measured scan coordinate, use:

`y_corrected = y_scan / cos(tilt_angle)`

The default tilt angle is 70 degrees.

In [ ]:
def apply_vertical_tilt_correction(df, tilt_angle_deg=70.0):
    out = df.copy()
    out['x_corr'] = out['x'].astype(float)
    out['y_corr'] = out['y'].astype(float) / np.cos(np.deg2rad(tilt_angle_deg))
    out.attrs.update(df.attrs)
    out.attrs['tilt_angle_deg'] = tilt_angle_deg
    return out


ebsd_corr = apply_vertical_tilt_correction(ebsd, tilt_angle_deg=70.0)

fig, ax = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
ax[0].scatter(ebsd_corr['x'], ebsd_corr['y'], s=1, c=ebsd_corr['IQ'], cmap='gray')
ax[0].set_title('Original scan coordinates')
ax[0].set_aspect('equal')
ax[1].scatter(ebsd_corr['x_corr'], ebsd_corr['y_corr'], s=1, c=ebsd_corr['IQ'], cmap='gray')
ax[1].set_title('Tilt-corrected coordinates')
ax[1].set_aspect('equal')
for a in ax:
    a.set_xlabel('x')
    a.set_ylabel('y')
plt.show()


## 5. Create Artificial Overlapping Tiles

This section cuts two neighboring overlapping tiles from one larger EBSD map. Tile B receives an artificial affine distortion plus optional nonlinear drift. The known transform is saved as ground truth.

In [ ]:
tile_width = None      # set manually, or leave as None for automatic sizing
tile_height = None
overlap_fraction = 0.25

artificial_translation = np.array([8.0, -4.0])
artificial_rotation_deg = 2.0
artificial_shear = 0.015
use_nonlinear_drift = True


def affine_matrix(rotation_deg=0.0, shear=0.0, scale=(1.0, 1.0)):
    a = np.deg2rad(rotation_deg)
    R = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
    Sh = np.array([[1.0, shear], [0.0, 1.0]])
    Sc = np.diag(scale)
    return R @ Sh @ Sc


def create_artificial_tiles(df, tile_width=None, tile_height=None, overlap_fraction=0.25,
                            translation=np.array([8.0, -4.0]), rotation_deg=2.0,
                            shear=0.015, nonlinear_drift=True):
    x0, x1 = df['x_corr'].min(), df['x_corr'].max()
    y0, y1 = df['y_corr'].min(), df['y_corr'].max()
    span_x, span_y = x1 - x0, y1 - y0
    tw = tile_width or span_x * 0.58
    th = tile_height or span_y * 0.90
    step = tw * (1 - overlap_fraction)
    ay0 = y0 + 0.05 * span_y
    ax0 = x0 + 0.02 * span_x
    bx0 = ax0 + step

    tile_a = df[(df.x_corr >= ax0) & (df.x_corr <= ax0 + tw) & (df.y_corr >= ay0) & (df.y_corr <= ay0 + th)].copy()
    tile_b = df[(df.x_corr >= bx0) & (df.x_corr <= bx0 + tw) & (df.y_corr >= ay0) & (df.y_corr <= ay0 + th)].copy()

    tile_a['tile'] = 'A'
    tile_b['tile'] = 'B'
    tile_a['x_tile_raw'] = tile_a['x_corr']
    tile_a['y_tile_raw'] = tile_a['y_corr']
    tile_b['x_tile_raw'] = tile_b['x_corr']
    tile_b['y_tile_raw'] = tile_b['y_corr']

    center_b = tile_b[['x_corr', 'y_corr']].mean().to_numpy()
    M = affine_matrix(rotation_deg=rotation_deg, shear=shear)
    p = tile_b[['x_corr', 'y_corr']].to_numpy()
    p_dist = (p - center_b) @ M.T + center_b + translation
    if nonlinear_drift:
        yn = (p[:, 1] - p[:, 1].min()) / max(np.ptp(p[:, 1]), 1e-9)
        p_dist[:, 0] += 1.5 * np.sin(2*np.pi*yn)
        p_dist[:, 1] += 0.6 * yn**2
    tile_b['x_obs'] = p_dist[:, 0]
    tile_b['y_obs'] = p_dist[:, 1]
    tile_a['x_obs'] = tile_a['x_corr']
    tile_a['y_obs'] = tile_a['y_corr']

    truth = {'M_B_true': M, 't_B_true_about_center_plus_translation': translation,
             'center_B': center_b, 'rotation_deg': rotation_deg, 'shear': shear,
             'nonlinear_drift': nonlinear_drift, 'tile_width': tw, 'tile_height': th,
             'overlap_fraction': overlap_fraction}
    return tile_a.reset_index(drop=True), tile_b.reset_index(drop=True), truth


tile_a, tile_b, ground_truth = create_artificial_tiles(
    ebsd_corr, tile_width, tile_height, overlap_fraction,
    artificial_translation, artificial_rotation_deg, artificial_shear, use_nonlinear_drift
)
print('Tile sizes:', len(tile_a), len(tile_b))
print('Ground truth:', ground_truth)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(tile_a.x_obs, tile_a.y_obs, s=1, label='Tile A')
ax.scatter(tile_b.x_obs, tile_b.y_obs, s=1, label='Tile B distorted')
ax.set_aspect('equal')
ax.legend(markerscale=5)
ax.set_title('Artificial overlapping tiles')
plt.show()


## 6. Grain / Superpixel Segmentation

If a grain ID column exists, it can be used directly. Otherwise this notebook forms approximate grains by connecting neighboring pixels whose misorientation is below a threshold, typically 5 degrees. The connected components become grain labels.

For large maps, this simple segmentation is deliberately conservative and may be slower than production EBSD software. It is included to keep the workflow open and reproducible.

In [ ]:
def infer_grid_shape(df, x_col='x_tile_raw', y_col='y_tile_raw'):
    xs = np.sort(df[x_col].unique())
    ys = np.sort(df[y_col].unique())
    return xs, ys


def segment_grains(df, misorientation_threshold_deg=5.0, use_existing=True):
    out = df.copy()
    existing = [c for c in ['grain_id', 'grain_id_true', 'GrainID'] if c in out.columns]
    if use_existing and existing:
        out['grain_id_seg'] = out[existing[0]].astype(int)
        return out

    xs, ys = infer_grid_shape(out)
    nx, ny = len(xs), len(ys)
    if nx * ny != len(out):
        # Fallback when data are not a complete rectangular grid: spatial KMeans superpixels.
        n_clusters = max(8, min(120, len(out) // 80))
        features = out[['x_tile_raw', 'y_tile_raw', 'phi1', 'Phi', 'phi2']].to_numpy()
        features = (features - features.mean(axis=0)) / (features.std(axis=0) + 1e-9)
        out['grain_id_seg'] = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=5).fit_predict(features)
        return out

    work = out.sort_values(['y_tile_raw', 'x_tile_raw']).reset_index()
    mats = euler_to_matrices(work[['phi1', 'Phi', 'phi2']].to_numpy()).reshape(ny, nx, 3, 3)
    parent = np.arange(nx * ny)

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for j in range(ny):
        for i in range(nx):
            idx = j * nx + i
            if i + 1 < nx and misorientation_deg(mats[j, i], mats[j, i+1]) < misorientation_threshold_deg:
                union(idx, idx + 1)
            if j + 1 < ny and misorientation_deg(mats[j, i], mats[j+1, i]) < misorientation_threshold_deg:
                union(idx, idx + nx)

    labels = np.array([find(i) for i in range(nx * ny)])
    labels = pd.factorize(labels)[0]
    work['grain_id_seg'] = labels
    out.loc[work['index'], 'grain_id_seg'] = work['grain_id_seg'].to_numpy()
    out['grain_id_seg'] = out['grain_id_seg'].astype(int)
    return out


def circular_mean_rad(a):
    return math.atan2(np.sin(a).mean(), np.cos(a).mean()) % (2*np.pi)


def grain_table(df):
    rows = []
    for gid, gdf in df.groupby('grain_id_seg'):
        euler_mean = np.array([
            circular_mean_rad(gdf['phi1'].to_numpy()),
            float(gdf['Phi'].mean()),
            circular_mean_rad(gdf['phi2'].to_numpy()),
        ])
        R = euler_to_matrix(*euler_mean)
        q = matrix_to_quaternion(R)
        rows.append({
            'grain_id': int(gid), 'area': len(gdf),
            'centroid_x': gdf['x_obs'].mean(), 'centroid_y': gdf['y_obs'].mean(),
            'centroid_x_raw': gdf['x_tile_raw'].mean(), 'centroid_y_raw': gdf['y_tile_raw'].mean(),
            'mean_IQ': gdf['IQ'].mean(), 'mean_CI': gdf['CI'].mean(),
            'phase': int(round(gdf['phase'].mode().iloc[0])) if 'phase' in gdf else 1,
            'texture_bin': texture_bin_from_euler(euler_mean[0], euler_mean[1], euler_mean[2]),
            'phi1': euler_mean[0], 'Phi': euler_mean[1], 'phi2': euler_mean[2],
            'R': R, 'q0': q[0], 'q1': q[1], 'q2': q[2], 'q3': q[3],
        })
    return pd.DataFrame(rows)


tile_a_seg = segment_grains(tile_a, misorientation_threshold_deg=5.0)
tile_b_seg = segment_grains(tile_b, misorientation_threshold_deg=5.0)
grains_a = grain_table(tile_a_seg)
grains_b = grain_table(tile_b_seg)
print('Segmented grains:', len(grains_a), len(grains_b))
display(grains_a.head())



## 7. Build Grain-Boundary Graph

Nodes are grains. Edges connect neighboring grains that share a boundary. Node features include mean orientation, centroid, area, IQ, and CI. Edge features include misorientation, centroid distance, and an approximate boundary length.

In [ ]:
def boundary_edges_from_pixels(df):
    xs, ys = infer_grid_shape(df)
    nx, ny = len(xs), len(ys)
    if nx * ny != len(df):
        return {}
    work = df.sort_values(['y_tile_raw', 'x_tile_raw']).reset_index(drop=True)
    labels2d = work['grain_id_seg'].to_numpy().reshape(ny, nx)
    edge_counts = {}
    for a, b in [(labels2d[:, :-1], labels2d[:, 1:]), (labels2d[:-1, :], labels2d[1:, :])]:
        mask = a != b
        for u, v in zip(a[mask], b[mask]):
            key = tuple(sorted((int(u), int(v))))
            edge_counts[key] = edge_counts.get(key, 0) + 1
    return edge_counts


def build_grain_graph(df, grains):
    G = nx.Graph()
    lookup = grains.set_index('grain_id')
    for row in grains.to_dict('records'):
        attrs = {k: v for k, v in row.items() if k != 'R'}
        attrs['R'] = row['R']
        G.add_node(int(row['grain_id']), **attrs)
    for (u, v), boundary_length in boundary_edges_from_pixels(df).items():
        if u not in lookup.index or v not in lookup.index:
            continue
        ru, rv = lookup.loc[u], lookup.loc[v]
        d = np.linalg.norm([ru.centroid_x - rv.centroid_x, ru.centroid_y - rv.centroid_y])
        mis = misorientation_deg(ru.R, rv.R, getattr(ru, 'phase', None), getattr(rv, 'phase', None))
        G.add_edge(u, v, misorientation_deg=mis, centroid_distance=d, boundary_length=boundary_length)
    return G


G_a = build_grain_graph(tile_a_seg, grains_a)
G_b = build_grain_graph(tile_b_seg, grains_b)
print(G_a)
print(G_b)



## 8. Percolation-Style Fingerprint

For a misorientation threshold `theta_0`, keep graph edges with `misorientation < theta_0`. Sweeping `theta_0` gives a connectivity curve: number of connected components, largest-cluster fraction, spanning score, and cluster-size statistics. This is analogous to a percolation process on the grain-boundary graph.

In [ ]:
def percolation_metrics(G, theta0, side='horizontal'):
    H = nx.Graph()
    H.add_nodes_from(G.nodes(data=True))
    H.add_edges_from((u, v, d) for u, v, d in G.edges(data=True) if d['misorientation_deg'] < theta0)
    comps = [set(c) for c in nx.connected_components(H)]
    sizes = np.array([len(c) for c in comps], dtype=float) if comps else np.array([0.0])
    n = max(G.number_of_nodes(), 1)

    xs = np.array([G.nodes[i]['centroid_x'] for i in G.nodes]) if G.nodes else np.array([0])
    ys = np.array([G.nodes[i]['centroid_y'] for i in G.nodes]) if G.nodes else np.array([0])
    xmin, xmax = xs.min(), xs.max()
    ymin, ymax = ys.min(), ys.max()
    tol_x = 0.08 * max(xmax - xmin, 1e-9)
    tol_y = 0.08 * max(ymax - ymin, 1e-9)
    spanning = 0.0
    for c in comps:
        cx = np.array([G.nodes[i]['centroid_x'] for i in c])
        cy = np.array([G.nodes[i]['centroid_y'] for i in c])
        if side == 'horizontal':
            spans = (cx.min() <= xmin + tol_x) and (cx.max() >= xmax - tol_x)
        else:
            spans = (cy.min() <= ymin + tol_y) and (cy.max() >= ymax - tol_y)
        spanning = max(spanning, float(spans) * len(c) / n)

    return {
        'theta0': theta0,
        'number_components': len(comps),
        'largest_cluster_fraction': sizes.max() / n,
        'mean_cluster_size': sizes.mean(),
        'std_cluster_size': sizes.std(),
        'spanning_score': spanning,
        'cluster_sizes': sizes,
    }


def percolation_sweep(G, thresholds=np.arange(1, 16, 1)):
    rows = [percolation_metrics(G, t) for t in thresholds]
    return pd.DataFrame(rows)


perc_a = percolation_sweep(G_a)
perc_b = percolation_sweep(G_b)

fig, ax = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
ax[0].plot(perc_a.theta0, perc_a.largest_cluster_fraction, '-o', label='A')
ax[0].plot(perc_b.theta0, perc_b.largest_cluster_fraction, '-o', label='B')
ax[0].set_ylabel('Largest cluster fraction')
ax[1].plot(perc_a.theta0, perc_a.number_components, '-o', label='A')
ax[1].plot(perc_b.theta0, perc_b.number_components, '-o', label='B')
ax[1].set_ylabel('Connected components')
for a in ax:
    a.set_xlabel('Misorientation threshold, deg')
    a.legend()
plt.show()

fingerprint_cols = ['largest_cluster_fraction', 'number_components', 'mean_cluster_size', 'std_cluster_size', 'spanning_score']
fingerprint_a = perc_a[fingerprint_cols].to_numpy().ravel()
fingerprint_b = perc_b[fingerprint_cols].to_numpy().ravel()
print('Fingerprint distance:', np.linalg.norm(fingerprint_a - fingerprint_b))


## 9. Boundary / Overlap Graph Matching with GATv2

This cell extracts grains near the right boundary of tile A and left boundary of tile B. Candidate matches are first scored with transparent EBSD features: geometry after approximate translation, orientation similarity, quality similarity, and local graph connectivity compatibility.

When `torch` and `torch_geometric` are installed, those candidate pairs are then passed through a small GATv2 graph neural network. The GATv2 encoder embeds each grain-boundary graph, and a pair MLP predicts match probability for candidate grain pairs. For artificial tiles, labels come from shared grain IDs. For real unlabeled data, the code falls back to pseudo-labels from the strongest heuristic candidates, so the notebook remains runnable but should be treated as weak supervision.


In [ ]:
def boundary_grains(grains, side, width_fraction=0.30):
    x = grains['centroid_x']
    width = x.max() - x.min()
    if side == 'right':
        return grains[x >= x.max() - width_fraction * width].copy()
    if side == 'left':
        return grains[x <= x.min() + width_fraction * width].copy()
    raise ValueError('side must be left or right')


def local_connectivity_signature(G, grain_id, theta0=10.0):
    neigh = list(G.neighbors(int(grain_id))) if int(grain_id) in G else []
    low = [n for n in neigh if G.edges[int(grain_id), n]['misorientation_deg'] < theta0]
    return np.array([len(neigh), len(low), np.mean([G.edges[int(grain_id), n]['boundary_length'] for n in neigh]) if neigh else 0.0])


DEFAULT_GATED_FUSION_CONFIG = {
    'MAX_MISORIENTATION_DEG': 10.0,
    'MAX_CENTROID_DISTANCE': 25.0,
    'MIN_CI': 0.05,
    'MIN_IQ': 0.0,
    'HEURISTIC_HIGH_CONF': 0.90,
    'HEURISTIC_LOW_CONF': 0.20,
    'GAT_HIGH_CONF': 0.90,
    'GAT_LOW_CONF': 0.20,
    'MATCH_THRESHOLD': 0.75,
    'STRONG_DISAGREEMENT': 0.50,
    'MAX_LINK_UNCERTAINTY': 0.65,
}



SELF_SUPERVISED_LINK_TRAINING = True
SELF_SUPERVISED_CONFIG = {
    'POSITIVE_HEURISTIC_SCORE': 0.82,
    'NEGATIVE_HEURISTIC_SCORE': 0.25,
    'MAX_POSITIVE_MISORIENTATION_DEG': 5.0,
    'MAX_POSITIVE_CENTROID_DISTANCE': 12.0,
    'MIN_NEGATIVE_MISORIENTATION_DEG': 12.0,
    'MIN_NEGATIVE_CENTROID_DISTANCE': 30.0,
    'POSITIVE_WEIGHT': 1.0,
    'NEGATIVE_WEIGHT': 0.7,
    'IGNORE_WEIGHT': 0.0,
}


def add_self_supervised_link_targets(candidates, config=None):
    """Create pseudo-labels from mutual high-confidence EBSD consistency.

    Ground-truth overlap labels are not used here. Positives must be mutual
    best heuristic candidates and pass strict orientation/geometry gates.
    Clear physics/geometry failures become negatives; ambiguous links get
    zero weight and do not contribute to the GATv2 loss.
    """
    config = dict(SELF_SUPERVISED_CONFIG if config is None else config)
    out = candidates.copy()
    if out.empty:
        out['ssl_label'] = []
        out['ssl_weight'] = []
        out['ssl_source'] = []
        return out

    out['ssl_label'] = 0.0
    out['ssl_weight'] = config['IGNORE_WEIGHT']
    out['ssl_source'] = 'ambiguous_unlabeled'

    best_b_for_a = out.groupby('grain_a')['heuristic_score'].idxmax()
    best_a_for_b = out.groupby('grain_b')['heuristic_score'].idxmax()
    mutual_idx = set(best_b_for_a).intersection(set(best_a_for_b))
    positive_mask = (
        out.index.isin(mutual_idx)
        & (out['heuristic_score'] >= config['POSITIVE_HEURISTIC_SCORE'])
        & (out['misorientation_deg'] <= config['MAX_POSITIVE_MISORIENTATION_DEG'])
        & (out['centroid_distance'] <= config['MAX_POSITIVE_CENTROID_DISTANCE'])
        & out.get('phase_match', True).astype(bool)
        & out.get('in_expected_overlap_region', True).astype(bool)
    )
    negative_mask = (
        (out['heuristic_score'] <= config['NEGATIVE_HEURISTIC_SCORE'])
        | (out['misorientation_deg'] >= config['MIN_NEGATIVE_MISORIENTATION_DEG'])
        | (out['centroid_distance'] >= config['MIN_NEGATIVE_CENTROID_DISTANCE'])
        | (~out.get('phase_match', True).astype(bool))
        | (~out.get('in_expected_overlap_region', True).astype(bool))
    )

    out.loc[negative_mask, ['ssl_label', 'ssl_weight', 'ssl_source']] = [0.0, config['NEGATIVE_WEIGHT'], 'physics_negative']
    out.loc[positive_mask, ['ssl_label', 'ssl_weight', 'ssl_source']] = [1.0, config['POSITIVE_WEIGHT'], 'mutual_consistency_positive']

    if out['ssl_label'].sum() == 0 and len(out):
        # Keep the training loop alive on difficult pairs without using labels:
        # the single strongest mutual candidate becomes a low-volume pseudo-positive.
        fallback_idx = out['heuristic_score'].idxmax()
        out.loc[fallback_idx, ['ssl_label', 'ssl_weight', 'ssl_source']] = [1.0, 0.5, 'fallback_top_candidate']
    return out


def estimate_link_uncertainty(candidate, config, final_score=0.0, gate_failed=False):
    """Estimate per-link uncertainty from model disagreement and distance to gates."""
    if gate_failed:
        return 1.0
    h = float(candidate.get('heuristic_score', 0.0))
    g = float(candidate.get('gatv2_probability', h))
    disagreement = abs(h - g)
    threshold = max(float(config.get('MATCH_THRESHOLD', 0.75)), 1e-6)
    threshold_risk = 1.0 - min(abs(float(final_score) - threshold) / threshold, 1.0)
    mis = float(candidate.get('misorientation_deg', config['MAX_MISORIENTATION_DEG']))
    dist = float(candidate.get('centroid_distance', config['MAX_CENTROID_DISTANCE']))
    ci = float(candidate.get('CI_mean', 0.0))
    mis_risk = np.clip(mis / max(config['MAX_MISORIENTATION_DEG'], 1e-6), 0.0, 1.0)
    dist_risk = np.clip(dist / max(config['MAX_CENTROID_DISTANCE'], 1e-6), 0.0, 1.0)
    quality_risk = 1.0 - np.clip(ci, 0.0, 1.0)
    uncertainty = 0.30*disagreement + 0.25*threshold_risk + 0.20*mis_risk + 0.15*dist_risk + 0.10*quality_risk
    return float(np.clip(uncertainty, 0.0, 1.0))


def add_link_uncertainty_columns(df, config=None):
    """Append uncertainty/confidence columns without changing decisions."""
    config = dict(DEFAULT_GATED_FUSION_CONFIG if config is None else config)
    out = df.copy()
    if out.empty:
        return out
    if 'gatv2_probability' not in out:
        out['gatv2_probability'] = out.get('heuristic_score', 0.0)
    out['heuristic_gatv2_disagreement'] = (out['heuristic_score'] - out['gatv2_probability']).abs()
    scores = out.get('final_score', out.get('heuristic_score', 0.0))
    out['link_uncertainty'] = [estimate_link_uncertainty(row, config, final_score=score) for row, score in zip(out.to_dict('records'), scores)]
    out['link_confidence'] = 1.0 - out['link_uncertainty']
    return out


def robust_one_to_one_candidates(candidates, score_col='final_score', label_col='final_label', max_uncertainty=None):
    """One-to-one assignment that rejects gated failures and high-uncertainty links before RANSAC."""
    if candidates.empty:
        return candidates.copy()
    max_uncertainty = DEFAULT_GATED_FUSION_CONFIG['MAX_LINK_UNCERTAINTY'] if max_uncertainty is None else max_uncertainty
    group_cols = ['tile_pair_id'] if 'tile_pair_id' in candidates.columns else [None]
    selected = []
    iterator = candidates.groupby('tile_pair_id') if group_cols != [None] else [(None, candidates)]
    for _, group in iterator:
        work = group.copy()
        if label_col in work:
            work = work[work[label_col] == 1]
        if 'decision_source' in work:
            work = work[work['decision_source'] != 'hard_gate_reject']
        if 'link_uncertainty' in work:
            work = work[work['link_uncertainty'] <= max_uncertainty]
        if work.empty:
            continue
        a_ids = sorted(work.grain_a.unique())
        b_ids = sorted(work.grain_b.unique())
        mat = np.ones((len(a_ids), len(b_ids))) * 1e6
        lookup = {}
        for idx, r in work.iterrows():
            uncertainty = float(r.get('link_uncertainty', 0.0))
            cost = -float(r[score_col]) + 0.15 * uncertainty
            i, j = a_ids.index(r.grain_a), b_ids.index(r.grain_b)
            if cost < mat[i, j]:
                mat[i, j] = cost
                lookup[(r.grain_a, r.grain_b)] = idx
        ri, ci = linear_sum_assignment(mat)
        for i, j in zip(ri, ci):
            if mat[i, j] < 1e5:
                idx = lookup.get((a_ids[i], b_ids[j]))
                if idx is not None:
                    selected.append(idx)
    out = candidates.loc[selected].copy() if selected else candidates.iloc[0:0].copy()
    if not out.empty:
        out['assignment_source'] = 'robust_hungarian_before_ransac'
    return out


def gated_fusion(candidate, config):
    """Fuse physics-aware EBSD heuristics with GATv2 probability.

    Hard gates reject physically implausible matches before ML can score them.
    This prevents the GNN from creating impossible seam links when orientation,
    geometry, phase, or data-quality constraints already rule out a candidate.
    """
    h = float(candidate.get('heuristic_score', 0.0))
    g = float(candidate.get('gatv2_probability', h))
    checks = [
        ('misorientation_exceeds_limit', float(candidate.get('misorientation_deg', np.inf)) > config['MAX_MISORIENTATION_DEG']),
        ('centroid_distance_exceeds_limit', float(candidate.get('centroid_distance', np.inf)) > config['MAX_CENTROID_DISTANCE']),
        ('CI_below_minimum', float(candidate.get('CI_mean', 0.0)) < config['MIN_CI']),
        ('IQ_below_minimum', float(candidate.get('IQ_mean', 0.0)) < config['MIN_IQ']),
        ('phase_mismatch', not bool(candidate.get('phase_match', True))),
        ('outside_expected_overlap_region', not bool(candidate.get('in_expected_overlap_region', True))),
    ]
    for reason, failed in checks:
        if failed:
            return {
                'final_score': 0.0,
                'final_label': 0,
                'decision_source': 'hard_gate_reject',
                'failed_gate_reason': reason,
                'heuristic_gatv2_disagreement': abs(h - g),
                'link_uncertainty': 1.0,
                'link_confidence': 0.0,
            }

    if h >= config['HEURISTIC_HIGH_CONF'] and g >= config['GAT_LOW_CONF']:
        score, source = h, 'heuristic_high_conf'
    elif g >= config['GAT_HIGH_CONF'] and h >= config['HEURISTIC_LOW_CONF']:
        score, source = g, 'gatv2_high_conf'
    elif h < config['HEURISTIC_LOW_CONF'] and g < config['GAT_HIGH_CONF']:
        score, source = 0.0, 'both_low_or_conflict_reject'
    elif abs(h - g) > config['STRONG_DISAGREEMENT']:
        score, source = min(h, g), 'conflict_conservative'
    else:
        score, source = 0.5*h + 0.5*g, 'fallback_average'

    uncertainty = estimate_link_uncertainty(candidate, config, final_score=score)
    final_label = int(score >= config['MATCH_THRESHOLD'] and uncertainty <= config.get('MAX_LINK_UNCERTAINTY', 1.0))
    return {
        'final_score': float(score),
        'final_label': final_label,
        'decision_source': source if final_label else 'uncertainty_reject',
        'failed_gate_reason': '' if final_label else 'link_uncertainty_exceeds_limit',
        'heuristic_gatv2_disagreement': abs(h - g),
        'link_uncertainty': uncertainty,
        'link_confidence': 1.0 - uncertainty,
    }


def apply_gated_fusion_df(candidates, config=None):
    """Apply gated fusion row-wise and append final score/decision columns."""
    config = dict(DEFAULT_GATED_FUSION_CONFIG if config is None else config)
    if candidates.empty:
        return candidates.copy()
    decisions = [gated_fusion(row, config) for row in candidates.to_dict('records')]
    return pd.concat([candidates.reset_index(drop=True), pd.DataFrame(decisions)], axis=1)


def graph_to_pyg_data(G):
    """Convert a NetworkX grain graph to PyG Data plus node-id bookkeeping."""
    import torch
    from torch_geometric.data import Data

    node_ids = list(G.nodes())
    node_index = {n: i for i, n in enumerate(node_ids)}
    x = []
    for n in node_ids:
        d = G.nodes[n]
        x.append([
            d['centroid_x'], d['centroid_y'], d['area'], d['mean_IQ'], d['mean_CI'],
            d['q0'], d['q1'], d['q2'], d['q3'], G.degree[n]
        ])
    x = np.asarray(x, dtype=np.float32)
    edges = []
    for u, v in G.edges():
        edges.append([node_index[u], node_index[v]])
        edges.append([node_index[v], node_index[u]])
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous() if edges else torch.empty((2, 0), dtype=torch.long)
    return Data(x=torch.tensor(x, dtype=torch.float32), edge_index=edge_index), node_ids


GATV2_AVAILABLE = OPTIONAL['torch'] and OPTIONAL['torch_geometric']
if GATV2_AVAILABLE:
    import torch
    import torch.nn.functional as F
    from torch import nn
    from torch_geometric.nn import GATv2Conv

    class GrainGATv2Matcher(nn.Module):
        """Two-tower GATv2 encoder with an MLP candidate-pair matcher."""
        def __init__(self, in_channels, pair_channels, hidden_channels=48, heads=3):
            super().__init__()
            self.gat1 = GATv2Conv(in_channels, hidden_channels, heads=heads, concat=True)
            self.gat2 = GATv2Conv(hidden_channels * heads, hidden_channels, heads=1, concat=False)
            self.match_mlp = nn.Sequential(
                nn.Linear(2 * hidden_channels + pair_channels, hidden_channels),
                nn.ReLU(),
                nn.Dropout(0.10),
                nn.Linear(hidden_channels, 1),
            )

        def encode(self, data):
            x = F.elu(self.gat1(data.x, data.edge_index))
            return self.gat2(x, data.edge_index)

        def forward(self, data_a, data_b, candidate_pairs, pair_features):
            za = self.encode(data_a)
            zb = self.encode(data_b)
            h = torch.cat([za[candidate_pairs[:, 0]], zb[candidate_pairs[:, 1]], pair_features], dim=1)
            return self.match_mlp(h).squeeze(-1)


def train_gatv2_candidate_scores(candidates, G_a, G_b, epochs=180, lr=0.01):
    """Train GATv2 on artificial-pair labels or heuristic pseudo-labels and return probabilities."""
    if not GATV2_AVAILABLE or candidates.empty:
        return None

    torch.manual_seed(SEED)
    data_a, node_ids_a = graph_to_pyg_data(G_a)
    data_b, node_ids_b = graph_to_pyg_data(G_b)
    id_to_idx_a = {gid: i for i, gid in enumerate(node_ids_a)}
    id_to_idx_b = {gid: i for i, gid in enumerate(node_ids_b)}

    combined = torch.cat([data_a.x, data_b.x], dim=0)
    mean = combined.mean(dim=0, keepdim=True)
    std = combined.std(dim=0, keepdim=True).clamp_min(1e-6)
    data_a.x = (data_a.x - mean) / std
    data_b.x = (data_b.x - mean) / std
    data_a = data_a.to(DEVICE)
    data_b = data_b.to(DEVICE)

    valid = candidates[candidates.grain_a.isin(id_to_idx_a) & candidates.grain_b.isin(id_to_idx_b)].copy()
    if valid.empty:
        return None

    candidate_pairs = torch.tensor(
        [[id_to_idx_a[int(r.grain_a)], id_to_idx_b[int(r.grain_b)]] for _, r in valid.iterrows()],
        dtype=torch.long, device=DEVICE,
    )
    pair_features_np = valid[[
        'geometry_score', 'orientation_score', 'connectivity_score', 'quality_score', 'phase_match', 'misorientation_deg'
    ]].to_numpy(dtype=np.float32).copy()
    pair_features_np[:, 5] = np.clip(pair_features_np[:, 5] / 15.0, 0.0, 1.0)
    pair_features = torch.tensor(pair_features_np, dtype=torch.float32, device=DEVICE)

    valid = add_self_supervised_link_targets(valid)
    train_mask_np = valid['ssl_weight'].to_numpy(dtype=np.float32) > 0
    if train_mask_np.sum() < 2 or valid.loc[train_mask_np, 'ssl_label'].nunique() < 2:
        print('Not enough self-supervised positives/negatives for GATv2; using heuristic probabilities.')
        return None
    train_mask = torch.tensor(train_mask_np, dtype=torch.bool, device=DEVICE)
    labels = torch.tensor(valid['ssl_label'].to_numpy(dtype=np.float32), dtype=torch.float32, device=DEVICE)
    sample_weights = torch.tensor(valid['ssl_weight'].to_numpy(dtype=np.float32), dtype=torch.float32, device=DEVICE)

    model = GrainGATv2Matcher(data_a.num_node_features, pair_features.shape[1]).to(DEVICE)
    train_labels = labels[train_mask]
    pos_weight = ((len(train_labels) - train_labels.sum()).clamp_min(1.0) / train_labels.sum().clamp_min(1.0)).reshape(1).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction='none')

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        logits = model(data_a, data_b, candidate_pairs, pair_features)
        loss_all = loss_fn(logits, labels)
        loss = (loss_all[train_mask] * sample_weights[train_mask]).sum() / sample_weights[train_mask].sum().clamp_min(1.0)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(data_a, data_b, candidate_pairs, pair_features)).cpu().numpy()

    out = pd.Series(index=valid.index, data=probs, name='gatv2_probability')
    print(f'GATv2 matcher trained self-supervised; pseudo_positives={int(train_labels.sum().item())}, weighted_candidates={int(train_mask.sum().item())}')
    return out


def match_boundary_grains(grains_a, grains_b, G_a, G_b, max_candidates=200,
                          weights=(0.45, 0.40, 0.15), use_gatv2=True):
    A = boundary_grains(grains_a, 'right')
    B = boundary_grains(grains_b, 'left')
    approx_shift = B[['centroid_x', 'centroid_y']].mean().to_numpy() - A[['centroid_x', 'centroid_y']].mean().to_numpy()
    rows = []
    geom_scale = max(np.std(np.r_[A.centroid_x, B.centroid_x]), np.std(np.r_[A.centroid_y, B.centroid_y]), 1.0)
    iq_scale = max(np.std(np.r_[A.mean_IQ, B.mean_IQ]), 1.0)
    for _, a in A.iterrows():
        pa = np.array([a.centroid_x, a.centroid_y]) + approx_shift
        sig_a = local_connectivity_signature(G_a, a.grain_id)
        for _, b in B.iterrows():
            pb = np.array([b.centroid_x, b.centroid_y])
            centroid_distance = float(np.linalg.norm(pa - pb))
            geom_score = np.exp(-centroid_distance / geom_scale)
            orient_mis = misorientation_deg(a.R, b.R, getattr(a, 'phase', None), getattr(b, 'phase', None))
            orientation_score = np.exp(-orient_mis / 5.0)
            quality_score = np.exp(-abs(a.mean_IQ - b.mean_IQ) / iq_scale) * np.exp(-abs(a.mean_CI - b.mean_CI) / 0.25)
            sig_b = local_connectivity_signature(G_b, b.grain_id)
            connectivity_score = np.exp(-np.linalg.norm(sig_a - sig_b) / 5.0)
            heuristic_score = weights[0]*geom_score + weights[1]*orientation_score + weights[2]*(0.5*quality_score + 0.5*connectivity_score)
            rows.append({
                'grain_a': int(a.grain_id), 'grain_b': int(b.grain_id),
                'score': heuristic_score, 'heuristic_score': heuristic_score,
                'geometry_score': geom_score, 'orientation_score': orientation_score,
                'connectivity_score': connectivity_score, 'quality_score': quality_score,
                'misorientation_deg': orient_mis,
                'centroid_distance': centroid_distance,
                'CI_mean': 0.5 * (a.mean_CI + b.mean_CI),
                'IQ_mean': 0.5 * (a.mean_IQ + b.mean_IQ),
                'phase_a': int(getattr(a, 'phase', 1)), 'phase_b': int(getattr(b, 'phase', 1)),
                'phase_match': int(getattr(a, 'phase', 1)) == int(getattr(b, 'phase', 1)),
                'texture_a': getattr(a, 'texture_bin', texture_bin_from_row(a)),
                'texture_b': getattr(b, 'texture_bin', texture_bin_from_row(b)),
                'texture_pair': f"{getattr(a, 'texture_bin', texture_bin_from_row(a))}__{getattr(b, 'texture_bin', texture_bin_from_row(b))}",
                'in_expected_overlap_region': True,
                'ax': a.centroid_x, 'ay': a.centroid_y, 'bx': b.centroid_x, 'by': b.centroid_y,
                'ax_raw': a.centroid_x_raw, 'ay_raw': a.centroid_y_raw,
                'bx_raw': b.centroid_x_raw, 'by_raw': b.centroid_y_raw,
            })
    cand = pd.DataFrame(rows).sort_values('heuristic_score', ascending=False).head(max_candidates)
    if cand.empty:
        return cand

    cand['gatv2_probability'] = cand['heuristic_score']
    if use_gatv2 and GATV2_AVAILABLE:
        gat_scores = train_gatv2_candidate_scores(cand, G_a, G_b)
        if gat_scores is not None:
            cand.loc[gat_scores.index, 'gatv2_probability'] = gat_scores
    elif use_gatv2:
        print('GATv2 requested, but torch_geometric is not installed; using heuristic graph matching fallback.')

    cand = apply_gated_fusion_df(cand, DEFAULT_GATED_FUSION_CONFIG)
    cand['score'] = cand['final_score']

    matched = robust_one_to_one_candidates(cand, score_col='final_score', label_col='final_label').sort_values('score', ascending=False)
    return matched


matches = match_boundary_grains(grains_a, grains_b, G_a, G_b, use_gatv2=True)
print('Candidate matches:', len(matches))
display(matches.head(15))



## 10. Estimate Affine Transform

Matched grain centroids are used to estimate an affine transform from observed tile B coordinates back into tile A coordinates. RANSAC rejects outlier matches. The plot compares distorted and corrected alignment.

In [ ]:
def estimate_transform_from_matches(matches, min_samples=3, residual_threshold=5.0):
    if len(matches) < min_samples:
        raise ValueError('Need at least three matches for affine estimation')
    src = matches[['bx', 'by']].to_numpy()  # distorted B observed points
    dst = matches[['ax', 'ay']].to_numpy()  # A reference points
    model, inliers = ransac((src, dst), AffineTransform, min_samples=min_samples,
                            residual_threshold=residual_threshold, max_trials=500)
    return model, inliers


model_b_to_a, inliers = estimate_transform_from_matches(matches)
matches['inlier'] = inliers
print('Inliers:', int(inliers.sum()), '/', len(inliers))
print('Estimated B observed -> A transform matrix:')
print(model_b_to_a.params)

p_b = tile_b[['x_obs', 'y_obs']].to_numpy()
p_b_aligned = model_b_to_a(p_b)
tile_b_aligned = tile_b.copy()
tile_b_aligned['x_stitch'] = p_b_aligned[:, 0]
tile_b_aligned['y_stitch'] = p_b_aligned[:, 1]
tile_a_aligned = tile_a.copy()
tile_a_aligned['x_stitch'] = tile_a_aligned['x_obs']
tile_a_aligned['y_stitch'] = tile_a_aligned['y_obs']

fig, ax = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
ax[0].scatter(tile_a.x_obs, tile_a.y_obs, s=1, label='A')
ax[0].scatter(tile_b.x_obs, tile_b.y_obs, s=1, label='B distorted')
ax[0].set_title('Before alignment')
ax[1].scatter(tile_a_aligned.x_stitch, tile_a_aligned.y_stitch, s=1, label='A')
ax[1].scatter(tile_b_aligned.x_stitch, tile_b_aligned.y_stitch, s=1, label='B aligned')
ax[1].set_title('After RANSAC affine alignment')
for a in ax:
    a.set_aspect('equal')
    a.legend(markerscale=5)
plt.show()


## 11. Stitch Tiles and Visualize

The stitched map is formed by transforming tile B into tile A's coordinate frame. Visualizations include IQ, CI, a simple orientation-color map, and a seam-region highlight.

In [ ]:
stitched = pd.concat([tile_a_aligned.assign(tile='A'), tile_b_aligned.assign(tile='B')], ignore_index=True)

def orientation_rgb(df):
    rgb = np.column_stack([
        (df['phi1'] % (2*np.pi)) / (2*np.pi),
        df['Phi'] / np.pi,
        (df['phi2'] % (2*np.pi)) / (2*np.pi),
    ])
    return np.clip(rgb, 0, 1)


fig, ax = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
sc0 = ax[0,0].scatter(stitched.x_stitch, stitched.y_stitch, s=1, c=stitched.IQ, cmap='gray')
ax[0,0].set_title('Stitched IQ map')
plt.colorbar(sc0, ax=ax[0,0], fraction=0.046)
sc1 = ax[0,1].scatter(stitched.x_stitch, stitched.y_stitch, s=1, c=stitched.CI, cmap='viridis')
ax[0,1].set_title('Stitched CI map')
plt.colorbar(sc1, ax=ax[0,1], fraction=0.046)
ax[1,0].scatter(stitched.x_stitch, stitched.y_stitch, s=1, c=orientation_rgb(stitched))
ax[1,0].set_title('Orientation color, Euler RGB proxy')
for label, sub in stitched.groupby('tile'):
    ax[1,1].scatter(sub.x_stitch, sub.y_stitch, s=1, label=label)
ax[1,1].set_title('Seam region by tile')
ax[1,1].legend(markerscale=5)
for a in ax.ravel():
    a.set_aspect('equal')
    a.set_xlabel('x stitched')
    a.set_ylabel('y stitched')
plt.show()


## 12. Validation Metrics

Validation compares matched grain pairs and the known artificial tile construction. Metrics include RMS coordinate error for inlier grain matches, mean misorientation across matched grains, broken graph connections across the seam, and percolation fingerprint distance.

In [ ]:
def validation_metrics(matches, model, G_a, G_b, perc_a, perc_b):
    inl = matches[matches['inlier']].copy()
    pred = model(inl[['bx', 'by']].to_numpy())
    target = inl[['ax', 'ay']].to_numpy()
    rms = float(np.sqrt(np.mean(np.sum((pred - target)**2, axis=1)))) if len(inl) else np.nan
    mean_mis = float(inl['misorientation_deg'].mean()) if len(inl) else np.nan

    # Approximate broken seam connections: strong orientation matches that still have large residuals.
    residuals = np.sqrt(np.sum((pred - target)**2, axis=1)) if len(inl) else np.array([])
    broken = int(np.sum((inl['misorientation_deg'].to_numpy() < 5.0) & (residuals > 5.0))) if len(inl) else 0

    fp_a = perc_a[fingerprint_cols].to_numpy().ravel()
    fp_b = perc_b[fingerprint_cols].to_numpy().ravel()
    fp_dist = float(np.linalg.norm(fp_a - fp_b))
    fp_sim = float(1.0 / (1.0 + fp_dist))
    return {
        'rms_coordinate_error_overlap': rms,
        'mean_misorientation_error_matched_grains_deg': mean_mis,
        'broken_graph_connections_across_seam': broken,
        'percolation_fingerprint_distance': fp_dist,
        'percolation_fingerprint_similarity': fp_sim,
        'n_matches': int(len(matches)),
        'n_inlier_matches': int(matches['inlier'].sum()),
    }


metrics = validation_metrics(matches, model_b_to_a, G_a, G_b, perc_a, perc_b)
pd.Series(metrics)


## 13. GATv2 Model Notes

The active GATv2 matcher is defined and used in Section 9 so that its probabilities can drive the affine stitching step. This section summarizes whether the graph neural network path was active in the current run.


In [ ]:
if GATV2_AVAILABLE:
    print("GATv2 was available and used in boundary matching. See matches['gatv2_probability'].")
    display(matches[['grain_a', 'grain_b', 'heuristic_score', 'gatv2_probability', 'final_score', 'final_label', 'decision_source', 'failed_gate_reason']].head(10))
else:
    print('GATv2 was requested, but torch_geometric is unavailable in this kernel.')
    print('Install optional dependencies with: pip install torch torch-geometric')
    print('The notebook used the heuristic graph/percolation matcher as a fallback so the rest still runs.')


## 14. Complete GATv2 Validation and Evaluation Pipeline

This section treats EBSD graph matching as link prediction between grain nodes across tile pairs. It builds a small artificial multi-pair dataset from the EBSD map, splits by `tile_pair_id` to avoid leakage, trains GATv2 with validation loss and early stopping, then reports classification, ranking, calibration, stitching-level, and per-category metrics.

Important leakage rule: candidate links are never split independently. All candidate links from the same artificial overlapping tile pair stay in exactly one split.

In [ ]:
from copy import deepcopy
import json as json_lib
from collections import defaultdict
import re

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve,
)
from sklearn.calibration import calibration_curve

try:
    import torchmetrics
    TORCHMETRICS_AVAILABLE = True
except Exception:
    TORCHMETRICS_AVAILABLE = False

# -----------------------------------------------------------------------------
# Configurable cropped-tile folders
# -----------------------------------------------------------------------------
# Folder 2 is an external holdout from a different big ANG file. It must never
# be used for model fitting, threshold tuning, calibration fitting, or fusion
# tuning. This gives a much more honest estimate of generalization.
FILE1_FOLDER = str(COLAB_PROJECT_ROOT / 'data/file1_cropped') if USE_COLAB_DRIVE_PATHS else 'data/file1_cropped'
FILE2_FOLDER = str(COLAB_PROJECT_ROOT / 'data/file2_cropped') if USE_COLAB_DRIVE_PATHS else 'data/file2_cropped'

# Local fallback only makes this notebook runnable in this workspace. In your
# project, point FILE1_FOLDER and FILE2_FOLDER to your actual cropped folders.
if not Path(FILE1_FOLDER).exists() and Path('Cropped').exists():
    print(f"Configured FILE1_FOLDER={FILE1_FOLDER!r} not found; using local fallback 'Cropped'.")
    FILE1_FOLDER = 'Cropped'
if not Path(FILE2_FOLDER).exists() and Path('Cropped_alloy718').exists():
    print(f"Configured FILE2_FOLDER={FILE2_FOLDER!r} not found; using local fallback 'Cropped_alloy718'.")
    FILE2_FOLDER = 'Cropped_alloy718'

MAX_POINTS_PER_TILE_FOR_GRAPHS = 6000
MAX_CLUSTERS_PER_TILE = 45
MIN_CLUSTERS_PER_TILE = 8
POINTS_PER_CLUSTER = 180
MIN_SHARED_POINTS_FOR_TRUE_MATCH = 5
MIN_SHARED_FRACTION_FOR_TRUE_MATCH = 0.08

PAIR_FEATURE_COLUMNS = [
    'geometry_score', 'orientation_score', 'connectivity_score', 'quality_score',
    'phase_match', 'degree_balance', 'area_ratio', 'misorientation_deg',
]

plots_dir = (COLAB_PROJECT_ROOT / 'plots') if USE_COLAB_DRIVE_PATHS else Path('plots')
plots_dir.mkdir(parents=True, exist_ok=True)
outputs_dir = (COLAB_PROJECT_ROOT / 'outputs_right_down') if USE_COLAB_DRIVE_PATHS else Path('outputs_right_down')
outputs_plots_dir = outputs_dir / 'plots'
outputs_plots_dir.mkdir(parents=True, exist_ok=True)


def finite_float(x):
    """JSON-safe float conversion for metrics that may be undefined."""
    x = float(x)
    return x if np.isfinite(x) else None


def read_tile_manifest(folder):
    folder = Path(folder)
    manifest_path = folder / 'tile_manifest.csv'
    if manifest_path.exists():
        manifest = pd.read_csv(manifest_path)
    else:
        rows = []
        for path in sorted(folder.glob('*_clean.ang')):
            m = re.search(r'tile_r(\d+)_c(\d+)_clean\.ang$', path.name)
            if m:
                rows.append({'tile_name': path.name, 'row': int(m.group(1)), 'col': int(m.group(2)), 'distorted': False})
        manifest = pd.DataFrame(rows)
    manifest = manifest[manifest['tile_name'].astype(str).str.endswith('_clean.ang')].copy()
    if 'distorted' in manifest.columns:
        manifest = manifest[manifest['distorted'].astype(str).str.lower().isin(['false', '0', 'no'])]
    manifest['path'] = manifest['tile_name'].map(lambda n: str(folder / n))
    manifest = manifest[manifest['path'].map(lambda p: Path(p).exists())].copy()
    manifest['row'] = manifest['row'].astype(int)
    manifest['col'] = manifest['col'].astype(int)
    return manifest.sort_values(['row', 'col']).reset_index(drop=True)


def sample_tile_df(df, max_points=MAX_POINTS_PER_TILE_FOR_GRAPHS):
    if len(df) <= max_points:
        return df.copy()
    return df.sample(n=max_points, random_state=SEED).sort_values(['y', 'x']).reset_index(drop=True)


def cluster_tile_to_grains(tile_df, source_file, max_clusters=MAX_CLUSTERS_PER_TILE):
    """Cluster EBSD pixels into graph nodes when explicit grain IDs are absent."""
    df = sample_tile_df(tile_df)
    df = df.copy()
    df['point_key'] = list(zip(np.round(df['x'].to_numpy(), 4), np.round(df['y'].to_numpy(), 4)))
    n_clusters = int(np.clip(len(df) // POINTS_PER_CLUSTER, MIN_CLUSTERS_PER_TILE, max_clusters))
    n_clusters = min(n_clusters, len(df))
    features = df[['x', 'y', 'phi1', 'Phi', 'phi2']].to_numpy(dtype=float)
    features = (features - features.mean(axis=0)) / (features.std(axis=0) + 1e-9)
    features[:, :2] *= 1.25
    labels = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=5).fit_predict(features)
    df['grain_id_seg'] = labels

    rows = []
    members = {}
    for gid, gdf in df.groupby('grain_id_seg'):
        euler_mean = np.array([
            circular_mean_rad(gdf['phi1'].to_numpy()),
            float(gdf['Phi'].mean()),
            circular_mean_rad(gdf['phi2'].to_numpy()),
        ])
        R = euler_to_matrix(*euler_mean)
        q = matrix_to_quaternion(R)
        member_set = set(gdf['point_key'])
        members[int(gid)] = member_set
        rows.append({
            'grain_id': int(gid), 'area': len(gdf),
            'centroid_x': gdf['x'].mean(), 'centroid_y': gdf['y'].mean(),
            'centroid_x_raw': gdf['x'].mean(), 'centroid_y_raw': gdf['y'].mean(),
            'mean_IQ': gdf['IQ'].mean(), 'mean_CI': gdf['CI'].mean(),
            'phase': int(round(gdf['phase'].mode().iloc[0])) if 'phase' in gdf else 1,
            'phi1': euler_mean[0], 'Phi': euler_mean[1], 'phi2': euler_mean[2],
            'R': R, 'q0': q[0], 'q1': q[1], 'q2': q[2], 'q3': q[3],
            'source_file': source_file,
        })
    return pd.DataFrame(rows), members


def build_cluster_graph(grains, k_neighbors=4):
    """Build an approximate grain/cluster adjacency graph using centroid kNN."""
    G = nx.Graph()
    for row in grains.to_dict('records'):
        attrs = {k: v for k, v in row.items() if k != 'R'}
        attrs['R'] = row['R']
        G.add_node(int(row['grain_id']), **attrs)
    if len(grains) < 2:
        return G
    coords = grains[['centroid_x', 'centroid_y']].to_numpy()
    tree = cKDTree(coords)
    _, idxs = tree.query(coords, k=min(k_neighbors + 1, len(grains)))
    lookup = grains.set_index('grain_id')
    ids = grains['grain_id'].to_numpy()
    for i, neighs in enumerate(np.atleast_2d(idxs)):
        u = int(ids[i])
        for j in neighs[1:]:
            v = int(ids[int(j)])
            if u == v or G.has_edge(u, v):
                continue
            ru, rv = lookup.loc[u], lookup.loc[v]
            G.add_edge(u, v,
                       misorientation_deg=misorientation_deg(ru.R, rv.R, getattr(ru, 'phase', None), getattr(rv, 'phase', None)),
                       centroid_distance=float(np.linalg.norm(coords[i] - coords[int(j)])),
                       boundary_length=1.0)
    return G


def load_tile_graph(tile_path, cache):
    tile_path = str(tile_path)
    if tile_path in cache:
        return cache[tile_path]
    df = parse_ang(tile_path)
    grains, members = cluster_tile_to_grains(df, source_file=tile_path)
    G = build_cluster_graph(grains)
    data, node_ids = graph_to_pyg_data(G)
    cache[tile_path] = {'df': df, 'grains': grains, 'members': members, 'G': G, 'data': data, 'node_ids': node_ids}
    return cache[tile_path]


def adjacent_tile_pairs(manifest):
    """Pair every clean tile with immediate right and down neighbors from the same folder."""
    by_rc = {(int(r.row), int(r.col)): r for _, r in manifest.iterrows()}
    pairs = []
    for (row, col), tile in sorted(by_rc.items()):
        right = by_rc.get((row, col + 1))
        if right is not None:
            pairs.append((tile, right, 'horizontal'))
        down = by_rc.get((row + 1, col))
        if down is not None:
            pairs.append((tile, down, 'vertical'))
    return pairs


def boundary_grains_for_side(grains, side, width_fraction=0.35):
    if side in ('left', 'right'):
        return boundary_grains(grains, side, width_fraction=width_fraction)
    y = grains['centroid_y']
    height = y.max() - y.min()
    if side == 'bottom':
        return grains[y >= y.max() - width_fraction * height].copy()
    if side == 'top':
        return grains[y <= y.min() + width_fraction * height].copy()
    raise ValueError('side must be left, right, top, or bottom')


def candidate_links_from_tile_graphs(left, right, direction='horizontal', max_candidates=500):
    ga, gb = left['grains'], right['grains']
    G_a, G_b = left['G'], right['G']
    if ga.empty or gb.empty:
        return pd.DataFrame()
    if direction == 'horizontal':
        A = boundary_grains_for_side(ga, 'right', width_fraction=0.35)
        B = boundary_grains_for_side(gb, 'left', width_fraction=0.35)
    else:
        A = boundary_grains_for_side(ga, 'bottom', width_fraction=0.35)
        B = boundary_grains_for_side(gb, 'top', width_fraction=0.35)
    rows = []
    geom_scale = max(np.std(np.r_[A.centroid_x, B.centroid_x]), np.std(np.r_[A.centroid_y, B.centroid_y]), 1.0)
    for _, a in A.iterrows():
        sig_a = local_connectivity_signature(G_a, a.grain_id)
        for _, b in B.iterrows():
            centroid_distance = float(np.linalg.norm([a.centroid_x - b.centroid_x, a.centroid_y - b.centroid_y]))
            orient_mis = misorientation_deg(a.R, b.R, getattr(a, 'phase', None), getattr(b, 'phase', None))
            geometry_score = float(np.exp(-centroid_distance / geom_scale))
            orientation_score = float(np.exp(-orient_mis / 5.0))
            quality_score = float(np.exp(-abs(a.mean_IQ - b.mean_IQ) / max(np.std(np.r_[A.mean_IQ, B.mean_IQ]), 1.0)) *
                                  np.exp(-abs(a.mean_CI - b.mean_CI) / 0.25))
            sig_b = local_connectivity_signature(G_b, b.grain_id)
            connectivity_score = float(np.exp(-np.linalg.norm(sig_a - sig_b) / 5.0))
            phase_match = int(getattr(a, 'phase', 1)) == int(getattr(b, 'phase', 1))
            degree_a = G_a.degree[int(a.grain_id)] if int(a.grain_id) in G_a else 0
            degree_b = G_b.degree[int(b.grain_id)] if int(b.grain_id) in G_b else 0
            degree_balance = 1.0 - abs(degree_a - degree_b) / max(degree_a + degree_b, 1)
            area_ratio = min(float(a.area), float(b.area)) / max(float(a.area), float(b.area), 1.0)
            heuristic_score = 0.45*geometry_score + 0.40*orientation_score + 0.15*(0.5*quality_score + 0.5*connectivity_score)
            shared = len(left['members'].get(int(a.grain_id), set()) & right['members'].get(int(b.grain_id), set()))
            denom = max(min(len(left['members'].get(int(a.grain_id), set())), len(right['members'].get(int(b.grain_id), set()))), 1)
            shared_fraction = shared / denom
            label = int(shared >= MIN_SHARED_POINTS_FOR_TRUE_MATCH and shared_fraction >= MIN_SHARED_FRACTION_FOR_TRUE_MATCH)
            rows.append({
                'grain_a': int(a.grain_id), 'grain_b': int(b.grain_id), 'label': float(label),
                'heuristic_score': heuristic_score,
                'geometry_score': geometry_score, 'orientation_score': orientation_score,
                'connectivity_score': connectivity_score, 'quality_score': quality_score,
                'misorientation_deg': orient_mis, 'centroid_distance': centroid_distance,
                'CI_mean': 0.5*(a.mean_CI + b.mean_CI), 'IQ_mean': 0.5*(a.mean_IQ + b.mean_IQ),
                'phase_a': int(getattr(a, 'phase', 1)), 'phase_b': int(getattr(b, 'phase', 1)),
                'phase_match': phase_match,
                'texture_a': getattr(a, 'texture_bin', texture_bin_from_row(a)),
                'texture_b': getattr(b, 'texture_bin', texture_bin_from_row(b)),
                'texture_pair': f"{getattr(a, 'texture_bin', texture_bin_from_row(a))}__{getattr(b, 'texture_bin', texture_bin_from_row(b))}",
                'in_expected_overlap_region': True,
                'degree_a': degree_a,
                'degree_b': degree_b,
                'degree_balance': degree_balance, 'area_ratio': area_ratio,
                'area_a': a.area, 'area_b': b.area, 'ci_a': a.mean_CI, 'ci_b': b.mean_CI,
                'ax': a.centroid_x, 'ay': a.centroid_y, 'bx': b.centroid_x, 'by': b.centroid_y,
                'shared_points': shared, 'shared_fraction': shared_fraction,
            })
    cand = pd.DataFrame(rows)
    if cand.empty:
        return cand
    positives = cand[cand.label == 1]
    negatives = cand[cand.label == 0].sort_values('heuristic_score', ascending=False).head(max_candidates)
    cand = pd.concat([positives, negatives], ignore_index=True).drop_duplicates(['grain_a', 'grain_b'])
    return add_self_supervised_link_targets(cand)


def make_folder_pair_item(pair_id, left_row, right_row, source_folder, source_name, cache, direction):
    left = load_tile_graph(left_row.path, cache)
    right = load_tile_graph(right_row.path, cache)
    cand = candidate_links_from_tile_graphs(left, right, direction=direction)
    if cand.empty or cand.label.sum() == 0:
        return None
    id_to_idx_a = {gid: i for i, gid in enumerate(left['node_ids'])}
    id_to_idx_b = {gid: i for i, gid in enumerate(right['node_ids'])}
    cand = cand[cand.grain_a.isin(id_to_idx_a) & cand.grain_b.isin(id_to_idx_b)].copy()
    if cand.empty or cand.label.sum() == 0:
        return None
    candidate_pairs = torch.tensor([[id_to_idx_a[int(r.grain_a)], id_to_idx_b[int(r.grain_b)]] for _, r in cand.iterrows()], dtype=torch.long)
    pair_features_np = cand[PAIR_FEATURE_COLUMNS].astype(float).to_numpy(dtype=np.float32).copy()
    mis_idx = PAIR_FEATURE_COLUMNS.index('misorientation_deg')
    pair_features_np[:, mis_idx] = np.clip(pair_features_np[:, mis_idx] / 15.0, 0.0, 1.0)
    cand['source_folder'] = source_folder
    cand['source_file_A'] = str(left_row.path)
    cand['source_file_B'] = str(right_row.path)
    return {
        'pair_id': pair_id,
        'source_folder': source_folder,
        'source_name': source_name,
        'source_files': {str(left_row.path), str(right_row.path)},
        'data_a': left['data'], 'data_b': right['data'],
        'candidate_pairs': candidate_pairs,
        'pair_features': torch.tensor(pair_features_np, dtype=torch.float32),
        'labels': torch.tensor(cand.label.to_numpy(dtype=np.float32).copy(), dtype=torch.float32),
        'ssl_labels': torch.tensor(cand.ssl_label.to_numpy(dtype=np.float32).copy(), dtype=torch.float32),
        'ssl_weights': torch.tensor(cand.ssl_weight.to_numpy(dtype=np.float32).copy(), dtype=torch.float32),
        'candidates': cand.reset_index(drop=True),
        'G_a': left['G'], 'G_b': right['G'],
        'grains_a': left['grains'], 'grains_b': right['grains'],
        'direction': direction,
        'truth': {'source_folder': source_folder, 'left_tile': str(left_row.path), 'right_tile': str(right_row.path), 'direction': direction},
    }



def move_pair_item_to_device(item, device):
    """Move PyG Data objects and pair tensors to the selected CPU/GPU device."""
    if device is None:
        return item
    item['data_a'] = item['data_a'].to(device)
    item['data_b'] = item['data_b'].to(device)
    item['candidate_pairs'] = item['candidate_pairs'].to(device)
    item['pair_features'] = item['pair_features'].to(device)
    item['labels'] = item['labels'].to(device)
    item['ssl_labels'] = item['ssl_labels'].to(device)
    item['ssl_weights'] = item['ssl_weights'].to(device)
    return item


def move_items_to_device(items, device):
    for item in items:
        move_pair_item_to_device(item, device)
    return items

def load_cropped_folder_tile_pairs(folder, source_name):
    """Load all horizontal adjacent clean-tile pairs from one cropped ANG folder."""
    folder = Path(folder)
    manifest = read_tile_manifest(folder)
    raw_pairs = adjacent_tile_pairs(manifest)
    cache = {}
    items = []
    for n, (left, right, direction) in enumerate(raw_pairs):
        pair_id = f'{source_name}_{direction}_r{int(left.row)}_c{int(left.col)}__r{int(right.row)}_c{int(right.col)}'
        item = make_folder_pair_item(pair_id, left, right, str(folder), source_name, cache, direction)
        if item is not None:
            items.append(item)
    print(f'{source_name}: loaded {len(items)} usable tile pairs from {folder} ({len(raw_pairs)} adjacent clean right/down pairs found)')
    return items


if GATV2_AVAILABLE:
    file1_items = load_cropped_folder_tile_pairs(FILE1_FOLDER, source_name='file1')
    file2_final_test_items = load_cropped_folder_tile_pairs(FILE2_FOLDER, source_name='file2')
else:
    file1_items = []
    file2_final_test_items = []
    print('Skipping folder dataset build because torch_geometric is unavailable.')



### 14.1 Data Split

The split is performed by `tile_pair_id`, not by candidate link. This prevents leakage from overlapping candidate sets within the same artificial tile pair.

In [ ]:
def split_file1_train_validation(items, train_frac=0.80, seed=SEED):
    """Split Folder 1 tile pairs by tile_pair_id, never individual candidate links."""
    pair_ids = np.array([item['pair_id'] for item in items])
    local_rng = np.random.default_rng(seed)
    shuffled = pair_ids.copy()
    local_rng.shuffle(shuffled)
    n = len(shuffled)
    n_train = max(1, int(round(train_frac * n))) if n else 0
    if n_train >= n and n > 1:
        n_train = n - 1
    train_ids = set(shuffled[:n_train])
    val_ids = set(shuffled[n_train:])
    train = [item for item in items if item['pair_id'] in train_ids]
    val = [item for item in items if item['pair_id'] in val_ids]
    return train, val


train_items, val_items = split_file1_train_validation(file1_items, train_frac=0.80) if file1_items else ([], [])
final_test_items = list(file2_final_test_items)

train_pair_ids = {item['pair_id'] for item in train_items}
val_pair_ids = {item['pair_id'] for item in val_items}
final_test_pair_ids = {item['pair_id'] for item in final_test_items}
train_val_source_files = set().union(*(item['source_files'] for item in train_items + val_items)) if (train_items or val_items) else set()
file2_source_files = set().union(*(item['source_files'] for item in final_test_items)) if final_test_items else set()

# Leakage prevention: Folder 2 is an external test set from Big ANG File 2.
assert train_pair_ids.isdisjoint(final_test_pair_ids), 'Leakage: Folder 2 tile_pair_id appears in training.'
assert val_pair_ids.isdisjoint(final_test_pair_ids), 'Leakage: Folder 2 tile_pair_id appears in validation.'
assert train_val_source_files.isdisjoint(file2_source_files), 'Leakage: Folder 2 source file appears in train/validation.'
assert all(item['source_name'] == 'file1' for item in train_items + val_items), 'Train/validation must come only from FILE1_FOLDER.'
assert all(item['source_name'] == 'file2' for item in final_test_items), 'Final test must come only from FILE2_FOLDER.'

FILE2_USED_FOR_TUNING = False

# Move tensors and PyG graphs to the active device. On Colab H100 this places
# GATv2 training/inference on CUDA while metadata remains in pandas/networkx.
if GATV2_AVAILABLE and DEVICE is not None:
    move_items_to_device(train_items + val_items + final_test_items, DEVICE)

print('Train tile pairs from FILE1_FOLDER:', len(train_items), 'candidate links:', sum(len(i['labels']) for i in train_items))
print('Validation tile pairs from FILE1_FOLDER:', len(val_items), 'candidate links:', sum(len(i['labels']) for i in val_items))
print('Final test tile pairs from FILE2_FOLDER:', len(final_test_items), 'candidate links:', sum(len(i['labels']) for i in final_test_items))
print('Self-supervised training signal:', {
    'enabled': SELF_SUPERVISED_LINK_TRAINING,
    'train_active_links': int(sum((i['ssl_weights'] > 0).sum().detach().cpu() for i in train_items)) if train_items else 0,
    'train_pseudo_positives': int(sum(i['ssl_labels'][i['ssl_weights'] > 0].sum().detach().cpu() for i in train_items)) if train_items else 0,
    'validation_active_links': int(sum((i['ssl_weights'] > 0).sum().detach().cpu() for i in val_items)) if val_items else 0,
})
print('Split pair IDs:', {
    'train_file1': sorted(train_pair_ids)[:10],
    'validation_file1': sorted(val_pair_ids)[:10],
    'final_test_file2': sorted(final_test_pair_ids)[:10],
})


### 14.2 Training Loop with Validation and Early Stopping

Each epoch computes average binary cross-entropy loss on the training tile pairs and validation tile pairs. Early stopping uses validation loss with `patience = 10`, and the best model weights are restored before test evaluation.

In [ ]:
def normalize_graph_features(items):
    """Normalize node features using training split statistics only."""
    if not items:
        return None
    x_all = torch.cat([torch.cat([item['data_a'].x, item['data_b'].x], dim=0) for item in items], dim=0)
    mean = x_all.mean(dim=0, keepdim=True)
    std = x_all.std(dim=0, keepdim=True).clamp_min(1e-6)
    return mean, std


def apply_feature_normalization(items, mean, std):
    for item in items:
        item['data_a'].x = (item['data_a'].x - mean) / std
        item['data_b'].x = (item['data_b'].x - mean) / std


def training_targets_for_item(item):
    """Return labels and weights for GATv2 training.

    With SELF_SUPERVISED_LINK_TRAINING=True, labels from shared parent
    coordinates are not used in the loss; they remain only for reporting.
    """
    if SELF_SUPERVISED_LINK_TRAINING and 'ssl_labels' in item and 'ssl_weights' in item:
        return item['ssl_labels'], item['ssl_weights']
    return item['labels'], torch.ones_like(item['labels'])


def batch_loss(model, items, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    losses = []
    for item in items:
        if is_train:
            optimizer.zero_grad()
        logits = model(item['data_a'], item['data_b'], item['candidate_pairs'], item['pair_features'])
        labels, weights = training_targets_for_item(item)
        loss_all = loss_fn(logits, labels)
        active = weights > 0
        if not torch.any(active):
            continue
        loss = (loss_all[active] * weights[active]).sum() / weights[active].sum().clamp_min(1.0)
        if is_train:
            loss.backward()
            optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses)) if losses else np.nan


def train_with_validation(train_items, val_items, epochs=100, patience=10, lr=0.01):
    if not train_items or not val_items:
        return None, [], [], np.nan
    norm = normalize_graph_features(train_items)
    apply_feature_normalization(train_items + val_items + final_test_items, *norm)

    in_channels = train_items[0]['data_a'].num_node_features
    pair_channels = train_items[0]['pair_features'].shape[1]
    model = GrainGATv2Matcher(in_channels, pair_channels).to(DEVICE)
    target_parts = [training_targets_for_item(item) for item in train_items]
    labels_all = torch.cat([labels[w > 0] for labels, w in target_parts])
    if len(labels_all) == 0 or torch.unique(labels_all).numel() < 2:
        print('Training skipped: self-supervised target generator produced insufficient active positives/negatives.')
        return None, [], [], np.nan
    pos = labels_all.sum().clamp_min(1.0)
    neg = (len(labels_all) - labels_all.sum()).clamp_min(1.0)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=(neg / pos).reshape(1), reduction='none')
    print('Training mode:', 'self-supervised pseudo-labels' if SELF_SUPERVISED_LINK_TRAINING else 'supervised shared-overlap labels')
    print('Training targets:', {'active_links': int(len(labels_all)), 'pseudo_positives': int(labels_all.sum().detach().cpu())})
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    train_losses, val_losses = [], []
    best_val = float('inf')
    best_state = deepcopy(model.state_dict())
    stale_epochs = 0
    for epoch in range(1, epochs + 1):
        train_loss = batch_loss(model, train_items, loss_fn, optimizer=optimizer)
        with torch.no_grad():
            val_loss = batch_loss(model, val_items, loss_fn, optimizer=None)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        if val_loss < best_val - 1e-5:
            best_val = val_loss
            best_state = deepcopy(model.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1
        if epoch == 1 or epoch % 10 == 0:
            print(f'Epoch {epoch:03d}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}')
        if stale_epochs >= patience:
            print(f'Early stopping at epoch {epoch}; best validation loss={best_val:.4f}')
            break
    model.load_state_dict(best_state)
    return model, train_losses, val_losses, best_val


if GATV2_AVAILABLE and train_items and val_items and final_test_items:
    eval_model, train_losses, val_losses, best_val_loss = train_with_validation(train_items, val_items, epochs=100, patience=10, lr=0.01)
    if eval_model is not None:
        torch.save({'model_state_dict': eval_model.state_dict(), 'device': str(DEVICE), 'best_val_loss': best_val_loss, 'self_supervised': SELF_SUPERVISED_LINK_TRAINING}, outputs_dir / 'best_gatv2_model.pt')
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(train_losses, label='Train loss')
        ax.plot(val_losses, label='Validation loss')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Weighted BCE loss')
        ax.set_title('GATv2 self-supervised training vs validation loss' if SELF_SUPERVISED_LINK_TRAINING else 'GATv2 training vs validation loss')
        ax.legend()
        fig.tight_layout()
        fig.savefig(plots_dir / 'loss_curve.png', dpi=180)
        plt.show()
else:
    eval_model, train_losses, val_losses, best_val_loss = None, [], [], np.nan
    print('Training skipped because GATv2 is unavailable or a split is empty.')


### 14.3 Validation-Tuned Gated Fusion and Method Comparison

This section evaluates four scoring methods: heuristic only, GATv2 only, fixed weighted average, and gated fusion. Gated fusion thresholds are tuned on the validation split only. Folder 2 is used once at the end as the external final Folder 2 final test set.

In [ ]:
def collect_predictions(model, items):
    """Collect GATv2 probabilities and candidate metadata for a list of tile-pair items."""
    rows = []
    model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        for item in items:
            logits = model(item['data_a'], item['data_b'], item['candidate_pairs'], item['pair_features'])
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            df = item['candidates'].copy()
            df['gatv2_probability'] = probs
            df['probability'] = probs
            df['tile_pair_id'] = item['pair_id']
            df['pair_direction'] = item.get('direction', 'horizontal')
            rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def classification_metrics_from_scores(df, score_col='final_score', label_col='final_label'):
    """Classification metrics for link prediction. PR-AUC is emphasized for sparse positives."""
    y_true = df.label.astype(int).to_numpy()
    y_score = df[score_col].to_numpy()
    y_pred = df[label_col].astype(int).to_numpy()
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_score) if len(np.unique(y_true)) == 2 else np.nan,
        'pr_auc': average_precision_score(y_true, y_score) if len(np.unique(y_true)) == 2 else np.nan,
        'false_match_rate': fp / max(fp + tn, 1),
        'missed_match_rate': fn / max(fn + tp, 1),
        'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn),
    }


def score_predictions_for_method(pred_df, method, config):
    """Create final_score/final_label for one comparison method."""
    out = pred_df.copy().reset_index(drop=True)
    threshold = config.get('MATCH_THRESHOLD', 0.75)
    if method == 'heuristic_only':
        out['final_score'] = out['heuristic_score'].clip(0, 1)
        out['decision_source'] = 'heuristic_only'
        out['failed_gate_reason'] = ''
    elif method == 'gatv2_only':
        out['final_score'] = out['gatv2_probability'].clip(0, 1)
        out['decision_source'] = 'gatv2_only'
        out['failed_gate_reason'] = ''
    elif method == 'fixed_weighted_average':
        out['final_score'] = 0.35*out['heuristic_score'] + 0.65*out['gatv2_probability']
        out['decision_source'] = 'fixed_weighted_average'
        out['failed_gate_reason'] = ''
    elif method == 'gated_fusion':
        out = apply_gated_fusion_df(out, config)
    else:
        raise ValueError(f'Unknown method: {method}')
    if 'final_label' not in out:
        out['final_label'] = (out['final_score'] >= threshold).astype(int)
    if 'link_uncertainty' not in out:
        out = add_link_uncertainty_columns(out, config)
    return out


def link_prediction_metrics(pred_df, score_col='final_score', ks=(1, 5, 10)):
    """Ranking metrics for graph matching: rank candidate B nodes for each A node."""
    reciprocal_ranks = []
    hits = {k: [] for k in ks}
    for (_, grain_a), group in pred_df.groupby(['tile_pair_id', 'grain_a']):
        ranked = group.sort_values(score_col, ascending=False).reset_index(drop=True)
        positive_positions = np.flatnonzero(ranked.label.to_numpy() == 1)
        if len(positive_positions) == 0:
            continue
        rank = int(positive_positions[0]) + 1
        reciprocal_ranks.append(1.0 / rank)
        for k in ks:
            hits[k].append(float(rank <= k))
    out = {'MRR': float(np.mean(reciprocal_ranks)) if reciprocal_ranks else np.nan}
    for k in ks:
        out[f'Hits@{k}'] = float(np.mean(hits[k])) if hits[k] else np.nan
    return out


def one_to_one_predicted_matches(pred_df, score_col='final_score', label_col='final_label'):
    """Robust one-to-one seam assignment before RANSAC."""
    return robust_one_to_one_candidates(pred_df, score_col=score_col, label_col=label_col)


DEFAULT_SEAM_VALIDATION_CONFIG = {
    'MIN_SEAM_MATCHES_FOR_RANSAC': 3,
    'MIN_RANSAC_INLIERS': 3,
    'MIN_RANSAC_INLIER_FRACTION': 0.45,
    'RANSAC_RESIDUAL_THRESHOLD': 5.0,
    'USE_LOCAL_POLYNOMIAL_CORRECTION': True,
    'LOCAL_POLYNOMIAL_DEGREE': 2,
    'MIN_LOCAL_POLY_POINTS': 6,
    'MAX_MEAN_LINK_UNCERTAINTY': 0.65,
}


def polynomial_design_matrix(xy, degree=2):
    x = xy[:, 0]
    y = xy[:, 1]
    cols = [np.ones(len(x)), x, y]
    if degree >= 2:
        cols.extend([x*x, x*y, y*y])
    if degree >= 3:
        cols.extend([x*x*x, x*x*y, x*y*y, y*y*y])
    return np.column_stack(cols)


def apply_local_polynomial_correction(src_xy, dst_xy, affine_model, degree=2, ridge=1e-6):
    """Fit a seam-local polynomial residual correction after affine alignment."""
    affine_xy = affine_model(src_xy)
    center = affine_xy.mean(axis=0, keepdims=True)
    scale = affine_xy.std(axis=0, keepdims=True) + 1e-6
    X = polynomial_design_matrix((affine_xy - center) / scale, degree=degree)
    residual = dst_xy - affine_xy
    lhs = X.T @ X + ridge * np.eye(X.shape[1])
    coeff = np.linalg.solve(lhs, X.T @ residual)
    corrected = affine_xy + X @ coeff
    return corrected, {'degree': degree, 'center': center.ravel().tolist(), 'scale': scale.ravel().tolist()}


def validate_one_seam_transform(pair_selected, seam_config=None):
    seam_config = dict(DEFAULT_SEAM_VALIDATION_CONFIG if seam_config is None else seam_config)
    empty = pair_selected.iloc[0:0].copy()
    base = {
        'seam_rejected': True,
        'seam_reject_reason': '',
        'ransac_inliers': 0,
        'ransac_inlier_fraction': 0.0,
        'affine_rmse': None,
        'local_corrected_rmse': None,
        'local_polynomial_used': False,
        'mean_selected_uncertainty': finite_float(pair_selected['link_uncertainty'].mean()) if 'link_uncertainty' in pair_selected and len(pair_selected) else None,
    }
    if len(pair_selected) < seam_config['MIN_SEAM_MATCHES_FOR_RANSAC']:
        base['seam_reject_reason'] = 'too_few_assigned_matches'
        return empty, base
    if base['mean_selected_uncertainty'] is not None and base['mean_selected_uncertainty'] > seam_config['MAX_MEAN_LINK_UNCERTAINTY']:
        base['seam_reject_reason'] = 'mean_link_uncertainty_too_high'
        return empty, base
    try:
        src = pair_selected[['bx', 'by']].to_numpy()
        dst = pair_selected[['ax', 'ay']].to_numpy()
        model, inliers = ransac((src, dst), AffineTransform, min_samples=3,
                                residual_threshold=seam_config['RANSAC_RESIDUAL_THRESHOLD'], max_trials=300)
        inliers = np.asarray(inliers, dtype=bool)
        kept = pair_selected.loc[inliers].copy()
        inlier_fraction = float(inliers.mean()) if len(inliers) else 0.0
        base['ransac_inliers'] = int(inliers.sum())
        base['ransac_inlier_fraction'] = inlier_fraction
        if base['ransac_inliers'] < seam_config['MIN_RANSAC_INLIERS'] or inlier_fraction < seam_config['MIN_RANSAC_INLIER_FRACTION']:
            base['seam_reject_reason'] = 'weak_ransac_consensus'
            return empty, base

        pred_xy = model(kept[['bx', 'by']].to_numpy())
        true_xy = kept[['ax', 'ay']].to_numpy()
        affine_rmse = float(np.sqrt(np.mean(np.sum((pred_xy - true_xy)**2, axis=1))))
        corrected_rmse = affine_rmse
        used_poly = False
        if seam_config['USE_LOCAL_POLYNOMIAL_CORRECTION'] and len(kept) >= seam_config['MIN_LOCAL_POLY_POINTS']:
            corrected_xy, _ = apply_local_polynomial_correction(
                kept[['bx', 'by']].to_numpy(), true_xy, model,
                degree=seam_config['LOCAL_POLYNOMIAL_DEGREE'],
            )
            poly_rmse = float(np.sqrt(np.mean(np.sum((corrected_xy - true_xy)**2, axis=1))))
            if np.isfinite(poly_rmse) and poly_rmse <= affine_rmse:
                pred_xy = corrected_xy
                corrected_rmse = poly_rmse
                used_poly = True
        kept['ransac_residual'] = np.sqrt(np.sum((pred_xy - true_xy)**2, axis=1))
        kept['local_polynomial_used'] = used_poly
        base.update({
            'seam_rejected': False,
            'seam_reject_reason': '',
            'affine_rmse': affine_rmse,
            'local_corrected_rmse': corrected_rmse,
            'local_polynomial_used': used_poly,
        })
        return kept, base
    except Exception as exc:
        base['seam_reject_reason'] = f'transform_fit_failed:{type(exc).__name__}'
        return empty, base


def stitching_validation(pred_df, items, score_col='final_score', label_col='final_label', seam_config=None):
    """Validate fused links with robust assignment, RANSAC consensus, and local correction."""
    selected = one_to_one_predicted_matches(pred_df, score_col=score_col, label_col=label_col)
    rmse_values, affine_rmse_values, mis_values, continuity_values, broken_counts, kept_counts = [], [], [], [], [], []
    inlier_fractions, seam_records = [], []
    rejected_pair_ids = []
    local_poly_count = 0
    for item in items:
        pair_selected = selected[selected.tile_pair_id == item['pair_id']]
        all_pair = pred_df[pred_df.tile_pair_id == item['pair_id']]
        true_positive_grains = set(all_pair.loc[all_pair.label == 1, 'grain_a'])
        kept, seam_info = validate_one_seam_transform(pair_selected, seam_config=seam_config)
        seam_info['tile_pair_id'] = item['pair_id']
        seam_records.append(seam_info)
        if seam_info['seam_rejected']:
            rejected_pair_ids.append(item['pair_id'])
            rmse_values.append(np.nan)
            affine_rmse_values.append(np.nan)
            inlier_fractions.append(seam_info['ransac_inlier_fraction'])
            continuity_values.append(0.0)
            broken_counts.append(0)
            kept_counts.append(0)
            continue
        rmse_values.append(seam_info['local_corrected_rmse'])
        affine_rmse_values.append(seam_info['affine_rmse'])
        inlier_fractions.append(seam_info['ransac_inlier_fraction'])
        local_poly_count += int(seam_info['local_polynomial_used'])
        mis_values.extend(kept['misorientation_deg'].tolist())
        correct = int(kept.label.sum())
        continuity_values.append(correct / max(len(true_positive_grains), 1))
        broken_counts.append(int((kept.label == 0).sum()))
        kept_counts.append(len(kept))
    hard_rejected = int((pred_df.get('decision_source', pd.Series(index=pred_df.index, dtype=object)) == 'hard_gate_reject').sum())
    return {
        'rmse_stitching_error': finite_float(np.nanmean(rmse_values)) if rmse_values else None,
        'affine_only_rmse_stitching_error': finite_float(np.nanmean(affine_rmse_values)) if affine_rmse_values else None,
        'mean_misorientation_error': finite_float(np.mean(mis_values)) if mis_values else None,
        'grain_continuity_score': finite_float(np.mean(continuity_values)) if continuity_values else None,
        'broken_connections': int(np.sum(broken_counts)) if broken_counts else 0,
        'candidates_before_gates': int(len(pred_df)),
        'hard_gate_rejected': hard_rejected,
        'accepted_by_fusion': int((pred_df[label_col] == 1).sum()) if label_col in pred_df else 0,
        'selected_predicted_matches': int(len(selected)),
        'kept_after_ransac': int(np.sum(kept_counts)),
        'mean_ransac_inlier_fraction': finite_float(np.nanmean(inlier_fractions)) if inlier_fractions else None,
        'local_polynomial_seams': int(local_poly_count),
        'seams_rejected_weak_consensus': int(len(rejected_pair_ids)),
        'weak_consensus_pair_ids': rejected_pair_ids[:20],
        'mean_accepted_link_uncertainty': finite_float(pred_df.loc[pred_df[label_col] == 1, 'link_uncertainty'].mean()) if 'link_uncertainty' in pred_df and label_col in pred_df else None,
        'seam_consensus': seam_records,
    }


def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true).astype(float)
    y_prob = np.asarray(y_prob).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (y_prob >= lo) & (y_prob < hi if hi < 1 else y_prob <= hi)
        if not np.any(mask):
            continue
        ece += mask.mean() * abs(y_true[mask].mean() - y_prob[mask].mean())
    return float(ece)



### 14.4 Validation-Only Threshold Tuning

`tune_gated_fusion(validation_data, config_grid)` searches threshold combinations on validation links only. The objective prioritizes lower false match rate first, then higher PR-AUC, higher F1, and lower stitching RMSE. The final Folder 2 final test set is not used during tuning.

In [ ]:
def build_gated_config_grid():
    base = dict(DEFAULT_GATED_FUSION_CONFIG)
    grid = []
    for max_mis in [7.5, 10.0, 12.5]:
        for max_dist in [15.0, 25.0, 35.0]:
            for h_hi in [0.85, 0.90, 0.95]:
                for g_hi in [0.85, 0.90, 0.95]:
                    cfg = dict(base)
                    cfg.update({
                        'MAX_MISORIENTATION_DEG': max_mis,
                        'MAX_CENTROID_DISTANCE': max_dist,
                        'HEURISTIC_HIGH_CONF': h_hi,
                        'HEURISTIC_LOW_CONF': 0.20,
                        'GAT_HIGH_CONF': g_hi,
                        'GAT_LOW_CONF': 0.20,
                        'MATCH_THRESHOLD': 0.75,
                    })
                    grid.append(cfg)
    return grid


def tune_gated_fusion(validation_data, config_grid, validation_items=None):
    """Tune gated-fusion thresholds on validation data only; never call this on test data."""
    best_cfg, best_metrics, best_key = None, None, None
    for cfg in config_grid:
        scored = score_predictions_for_method(validation_data, 'gated_fusion', cfg)
        cls = classification_metrics_from_scores(scored)
        stitch = stitching_validation(scored, validation_items or [], score_col='final_score', label_col='final_label') if validation_items else {'rmse_stitching_error': None}
        rmse = stitch.get('rmse_stitching_error')
        rmse_key = rmse if rmse is not None else 1e9
        key = (cls['false_match_rate'], -cls['pr_auc'], -cls['f1'], rmse_key)
        if best_key is None or key < best_key:
            best_key = key
            best_cfg = cfg
            best_metrics = {'classification': cls, 'stitching': stitch}
    return best_cfg, best_metrics


if eval_model is not None:
    val_pred_df = collect_predictions(eval_model, val_items)
    assert not FILE2_USED_FOR_TUNING, 'FILE2_FOLDER was used before final test; aborting external test evaluation.'
    final_test_pred_df = collect_predictions(eval_model, final_test_items)
    gated_config_grid = build_gated_config_grid()
    tuned_gated_config, tuned_validation_metrics = tune_gated_fusion(val_pred_df, gated_config_grid, validation_items=val_items)
    print('Tuned gated fusion config from validation split only:')
    print(json_lib.dumps(tuned_gated_config, indent=2))
    print('Validation objective metrics:', tuned_validation_metrics)
else:
    val_pred_df = pd.DataFrame()
    final_test_pred_df = pd.DataFrame()
    tuned_gated_config = dict(DEFAULT_GATED_FUSION_CONFIG)
    tuned_validation_metrics = {}


### 14.5 Final Test Evaluation and Method Comparison

The tuned gated config is frozen here. The Folder 2 final test set is evaluated once for heuristic-only, GATv2-only, fixed weighted average, and gated fusion. Reported stitching metrics are recomputed after RANSAC rejects geometric outliers.

In [ ]:
def evaluate_method(pred_df, items, method, config):
    scored = score_predictions_for_method(pred_df, method, config)
    cls = classification_metrics_from_scores(scored)
    rank = link_prediction_metrics(scored, score_col='final_score')
    stitch = stitching_validation(scored, items, score_col='final_score', label_col='final_label')
    return scored, {'classification': cls, 'link_prediction': rank, 'stitching': stitch}


if not final_test_pred_df.empty:
    method_results = {}
    method_decisions = {}
    for method in ['heuristic_only', 'gatv2_only', 'fixed_weighted_average', 'gated_fusion']:
        scored, result = evaluate_method(final_test_pred_df, final_test_items, method, tuned_gated_config)
        method_decisions[method] = scored
        method_results[method] = result
        print('\n', method)
        print('classification:', result['classification'])
        print('stitching:', result['stitching'])

    gated_file2_df = method_decisions['gated_fusion'].copy()
    test_classification = method_results['gated_fusion']['classification']
    test_link_metrics = method_results['gated_fusion']['link_prediction']
    stitching_metrics = method_results['gated_fusion']['stitching']
else:
    method_results, method_decisions = {}, {}
    gated_file2_df = pd.DataFrame()
    test_classification, test_link_metrics, stitching_metrics = {}, {}, {}


### 14.6 Calibration, Per-Category Analysis, and Output Files

The calibration plot uses gated-fusion final scores. Decisions are exported for auditability, including the hard-gate reason and the final decision source.

In [ ]:
def safe_group_metrics(df):
    if df.empty:
        return {}
    return {k: (finite_float(v) if isinstance(v, (float, np.floating)) else v)
            for k, v in classification_metrics_from_scores(df).items()}


def per_category_analysis(pred_df):
    out = {'degree': {}, 'CI': {}, 'misorientation_difficulty': {}, 'phase': {}, 'texture': {}, 'uncertainty': {}}
    df = pred_df.copy()
    df['degree_category'] = np.where(df.degree_a <= df.degree_a.median(), 'small_degree', 'large_degree')
    df['ci_bin'] = pd.cut(df.CI_mean, bins=[-np.inf, 0.5, 0.75, np.inf], labels=['low_CI', 'mid_CI', 'high_CI'])
    df['misorientation_bin'] = pd.cut(df.misorientation_deg, bins=[-np.inf, 2, 5, 15, np.inf],
                                      labels=['easy_<2deg', 'moderate_2_5deg', 'hard_5_15deg', 'very_hard_>15deg'])
    if 'link_uncertainty' in df:
        df['uncertainty_bin'] = pd.cut(df.link_uncertainty, bins=[-np.inf, 0.25, 0.5, 0.75, np.inf],
                                       labels=['low_uncertainty', 'mid_uncertainty', 'high_uncertainty', 'reject_level_uncertainty'])
    for name, group in df.groupby('degree_category', observed=False):
        out['degree'][str(name)] = safe_group_metrics(group)
    for name, group in df.groupby('ci_bin', observed=False):
        out['CI'][str(name)] = safe_group_metrics(group)
    for name, group in df.groupby('misorientation_bin', observed=False):
        out['misorientation_difficulty'][str(name)] = safe_group_metrics(group)
    if 'phase_a' in df:
        for name, group in df.groupby('phase_a', observed=False):
            out['phase'][str(name)] = safe_group_metrics(group)
    if 'texture_a' in df:
        for name, group in df.groupby('texture_a', observed=False):
            out['texture'][str(name)] = safe_group_metrics(group)
    if 'uncertainty_bin' in df:
        for name, group in df.groupby('uncertainty_bin', observed=False):
            out['uncertainty'][str(name)] = safe_group_metrics(group)
    return out


if not gated_file2_df.empty:
    y_true = gated_file2_df.label.astype(int)
    y_score = gated_file2_df.final_score
    frac_pos, mean_pred = calibration_curve(y_true, y_score, n_bins=10, strategy='uniform')
    ece = expected_calibration_error(y_true, y_score, n_bins=10)
    category_metrics = per_category_analysis(gated_file2_df)

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(mean_pred, frac_pos, 'o-', label=f'ECE={ece:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Empirical match frequency')
    ax.set_title('Gated fusion calibration curve')
    ax.legend()
    fig.tight_layout()
    fig.savefig(outputs_plots_dir / 'calibration_curve.png', dpi=180)
    fig.savefig(plots_dir / 'calibration_curve.png', dpi=180)
    plt.show()

    # Comparison plot across methods.
    compare_df = pd.DataFrame({
        method: {
            'F1': res['classification']['f1'],
            'PR-AUC': res['classification']['pr_auc'],
            'False match rate': res['classification']['false_match_rate'],
            'Continuity': res['stitching']['grain_continuity_score'] if res['stitching']['grain_continuity_score'] is not None else np.nan,
        }
        for method, res in method_results.items()
    }).T
    fig, ax = plt.subplots(figsize=(9, 4.5))
    compare_df.plot(kind='bar', ax=ax)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Metric value')
    ax.set_title('Method comparison on held-out test tile pairs')
    ax.legend(loc='best')
    fig.tight_layout()
    fig.savefig(outputs_plots_dir / 'gated_fusion_comparison.png', dpi=180)
    plt.show()

    # Confusion matrix for gated fusion.
    cm = confusion_matrix(gated_file2_df.label.astype(int), gated_file2_df.final_label.astype(int), labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, cmap='Blues')
    for (i, j), val in np.ndenumerate(cm):
        ax.text(j, i, str(val), ha='center', va='center')
    ax.set_xticks([0, 1], labels=['Pred 0', 'Pred 1'])
    ax.set_yticks([0, 1], labels=['True 0', 'True 1'])
    ax.set_title('Gated fusion confusion matrix')
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(outputs_plots_dir / 'confusion_matrix_gated_fusion.png', dpi=180)
    plt.show()

    # ROC/PR plots for gated fusion for continuity with the previous notebook outputs.
    fpr, tpr, _ = roc_curve(y_true, y_score)
    precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_score)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, label=f"ROC-AUC={test_classification['roc_auc']:.3f}")
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax.set_xlabel('False positive rate')
    ax.set_ylabel('True positive rate')
    ax.set_title('Gated fusion ROC curve')
    ax.legend()
    fig.tight_layout()
    fig.savefig(plots_dir / 'ROC_curve.png', dpi=180)
    plt.show()

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(recall_curve, precision_curve, label=f"PR-AUC={test_classification['pr_auc']:.3f}")
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title('Gated fusion precision-recall curve')
    ax.legend()
    fig.tight_layout()
    fig.savefig(plots_dir / 'PR_curve.png', dpi=180)
    plt.show()
else:
    ece = np.nan
    category_metrics = {}



### 14.7 Logging, Output, and Summary

Saves the tuned gated config, method-comparison metrics, and per-candidate gated decisions. The decision CSV is designed for auditing why any candidate was accepted or rejected.

In [ ]:
if eval_model is not None and not gated_file2_df.empty:
    decisions_export = gated_file2_df.rename(columns={
        'grain_a': 'node_A',
        'grain_b': 'node_B',
        'label': 'true_label',
    })
    export_cols = [
        'tile_pair_id', 'node_A', 'node_B', 'true_label',
        'heuristic_score', 'gatv2_probability', 'final_score', 'final_label',
        'decision_source', 'failed_gate_reason', 'misorientation_deg',
        'centroid_distance', 'CI_mean', 'IQ_mean', 'phase_a', 'phase_b',
        'phase_match', 'texture_a', 'texture_b', 'texture_pair',
        'heuristic_gatv2_disagreement', 'link_uncertainty', 'link_confidence',
    ]
    decisions_csv = decisions_export[[c for c in export_cols if c in decisions_export.columns]]
    decisions_csv.to_csv(outputs_dir / 'gated_fusion_decisions.csv', index=False)
    decisions_csv.to_csv(outputs_dir / 'predictions_file2.csv', index=False)
    (outputs_dir / 'gated_fusion_config.json').write_text(json_lib.dumps(tuned_gated_config, indent=2))

    eval_summary = {
        'training_mode': 'self_supervised_pseudo_labels' if SELF_SUPERVISED_LINK_TRAINING else 'supervised_shared_overlap_labels',
        'self_supervised_config': SELF_SUPERVISED_CONFIG,
        'best_validation_loss': finite_float(best_val_loss),
        'tuned_gated_fusion_config': tuned_gated_config,
        'validation_tuning_metrics': tuned_validation_metrics,
        'methods': {
            method: {
                'classification': {k: finite_float(v) if isinstance(v, (float, np.floating)) else v for k, v in res['classification'].items()},
                'link_prediction': {k: finite_float(v) for k, v in res['link_prediction'].items()},
                'stitching': res['stitching'],
            }
            for method, res in method_results.items()
        },
        'classification': {k: finite_float(v) if isinstance(v, (float, np.floating)) else v for k, v in test_classification.items()},
        'link_prediction': {k: finite_float(v) for k, v in test_link_metrics.items()},
        'calibration': {'ECE': finite_float(ece)},
        'stitching': stitching_metrics,
        'per_category': category_metrics,
        'split_sizes': {
            'train_pairs': len(train_items), 'validation_pairs': len(val_items), 'final_test_pairs_file2': len(final_test_items),
            'train_links': sum(len(i['labels']) for i in train_items),
            'validation_links': sum(len(i['labels']) for i in val_items),
            'final_test_links_file2': sum(len(i['labels']) for i in final_test_items),
            'train_ssl_active_links': int(sum((i['ssl_weights'] > 0).sum().detach().cpu() for i in train_items)) if train_items else 0,
            'train_ssl_pseudo_positives': int(sum(i['ssl_labels'][i['ssl_weights'] > 0].sum().detach().cpu() for i in train_items)) if train_items else 0,
        },
        'note': 'GATv2 training uses self-supervised pseudo-labels when enabled. PR-AUC is emphasized because EBSD link prediction is usually class-imbalanced. Gated thresholds were tuned on validation data only.',
    }
    (outputs_dir / 'gated_fusion_metrics.json').write_text(json_lib.dumps(eval_summary, indent=2))
    (outputs_dir / 'final_test_metrics_file2.json').write_text(json_lib.dumps(eval_summary, indent=2))
    train_val_metrics = {'training_mode': eval_summary['training_mode'], 'self_supervised_config': SELF_SUPERVISED_CONFIG, 'best_validation_loss': finite_float(best_val_loss), 'train_losses': train_losses, 'val_losses': val_losses, 'train_pairs': len(train_items), 'validation_pairs': len(val_items), 'file1_folder': FILE1_FOLDER}
    (outputs_dir / 'train_val_metrics.json').write_text(json_lib.dumps(train_val_metrics, indent=2))
    Path('metrics.json').write_text(json_lib.dumps(eval_summary, indent=2))
    print('Saved gated fusion outputs under outputs/.')

    print('\nGated fusion evaluation summary')
    print('Best validation loss:', eval_summary['best_validation_loss'])
    print('Test accuracy:', eval_summary['classification']['accuracy'])
    print('F1 score:', eval_summary['classification']['f1'])
    print('ROC-AUC:', eval_summary['classification']['roc_auc'])
    print('PR-AUC:', eval_summary['classification']['pr_auc'])
    print('False match rate:', eval_summary['classification']['false_match_rate'])
    print('Missed match rate:', eval_summary['classification']['missed_match_rate'])
    print('RMSE stitching error:', eval_summary['stitching']['rmse_stitching_error'])
    print('Mean misorientation error:', eval_summary['stitching']['mean_misorientation_error'])
    print('Grain continuity score:', eval_summary['stitching']['grain_continuity_score'])
    print('Candidates before gates:', eval_summary['stitching']['candidates_before_gates'])
    print('Hard-gate rejected:', eval_summary['stitching']['hard_gate_rejected'])
    print('Accepted by gated fusion:', eval_summary['stitching']['accepted_by_fusion'])
    print('Kept after RANSAC:', eval_summary['stitching']['kept_after_ransac'])
else:
    eval_summary = {}
    print('No gated-fusion evaluation summary written because full GATv2 evaluation did not run.')



### 14.8 Parent Big ANG Final Validation

This optional final validation compares the held-out Folder 2 stitching result against the original parent Big ANG File 2. Folder 2 remains external: the parent file is used only after training, validation tuning, and final Folder 2 predictions are complete.

Set `PARENT_FILE2_ANG` to the original full `.ang` file that produced `FILE2_FOLDER`. This section does not train or tune anything.

In [ ]:
# Original parent ANG files. Only PARENT_FILE2_ANG is used for final external validation.
PARENT_FILE1_ANG = None
PARENT_FILE2_ANG = "EBSD stiching dev dataset/alloy718_aged_1000C_3700s-1_17a.ang"
RUN_PARENT_FILE2_VALIDATION = False  # Set True after paths and Folder 2 predictions are available.


def parent_lookup_tree(parent_df):
    """Build a nearest-coordinate lookup for the parent map."""
    parent = parent_df.copy()
    if 'x_corr' not in parent or 'y_corr' not in parent:
        parent['x_corr'] = parent['x']
        parent['y_corr'] = parent['y']
    coords = parent[['x_corr', 'y_corr']].to_numpy(dtype=float)
    return parent, cKDTree(coords)


def point_orientation_misorientation_deg(tile_rows, parent_rows):
    """Mean pointwise SO(3) misorientation between stitched tile rows and nearest parent rows."""
    if len(tile_rows) == 0:
        return np.nan
    tile_R = euler_to_matrices(tile_rows[['phi1', 'Phi', 'phi2']].to_numpy())
    parent_R = euler_to_matrices(parent_rows[['phi1', 'Phi', 'phi2']].to_numpy())
    vals = [misorientation_deg(a, b) for a, b in zip(tile_R, parent_R)]
    return float(np.mean(vals)) if vals else np.nan


def estimate_pair_transform_from_decisions(pair_decisions):
    """Estimate B->A affine transform from accepted final matches for one tile pair."""
    accepted = pair_decisions[pair_decisions['final_label'] == 1].copy()
    if len(accepted) < 3:
        return None, None
    try:
        model, inliers = ransac(
            (accepted[['bx', 'by']].to_numpy(), accepted[['ax', 'ay']].to_numpy()),
            AffineTransform,
            min_samples=3,
            residual_threshold=5.0,
            max_trials=200,
        )
        return model, np.asarray(inliers, dtype=bool)
    except Exception:
        return None, None


def get_folder2_predictions_for_parent_validation():
    """Use in-memory Folder 2 decisions or load outputs/predictions_file2.csv."""
    if 'gated_file2_df' in globals() and isinstance(gated_file2_df, pd.DataFrame) and not gated_file2_df.empty:
        return gated_file2_df.copy()
    csv_path = outputs_dir / 'predictions_file2.csv'
    if csv_path.exists():
        print(f'Loading Folder 2 predictions from {csv_path}')
        return pd.read_csv(csv_path)
    raise RuntimeError('Folder 2 predictions are missing. Run the final-test/gated-fusion evaluation cell first.')


def get_folder2_items_for_parent_validation():
    """Use in-memory final_test_items or rebuild Folder 2 tile-pair metadata."""
    if 'final_test_items' in globals() and final_test_items:
        return final_test_items
    if 'load_cropped_folder_tile_pairs' in globals():
        print('Rebuilding Folder 2 tile-pair metadata from FILE2_FOLDER.')
        return load_cropped_folder_tile_pairs(FILE2_FOLDER, source_name='file2')
    raise RuntimeError('Folder 2 tile-pair metadata is missing. Run the dataset loading cells first.')


def validate_file2_stitch_against_parent(parent_ang_path, final_test_items, decisions_df,
                                         max_points_per_pair=5000,
                                         nearest_tolerance=2.0):
    """Compare Folder 2 stitched pair coordinates/orientations against parent Big ANG File 2.

    This is a final audit only. It never feeds back into training, validation,
    calibration, gated-fusion tuning, or threshold selection.
    """
    parent_df = parse_ang(parent_ang_path)
    parent_df['x_corr'] = parent_df['x']
    parent_df['y_corr'] = parent_df['y']
    parent_df, tree = parent_lookup_tree(parent_df)

    coord_errors = []
    misorientation_values = []
    within_tol = []
    evaluated_pairs = 0
    skipped_pairs = 0

    item_by_id = {item['pair_id']: item for item in final_test_items}
    for pair_id, pair_decisions in decisions_df.groupby('tile_pair_id'):
        item = item_by_id.get(pair_id)
        if item is None:
            skipped_pairs += 1
            continue
        model, _ = estimate_pair_transform_from_decisions(pair_decisions)
        if model is None:
            skipped_pairs += 1
            continue

        left_path = item['truth']['left_tile']
        right_path = item['truth']['right_tile']
        left_df = parse_ang(left_path)
        right_df = parse_ang(right_path)
        left_df['x_stitch'] = left_df['x']
        left_df['y_stitch'] = left_df['y']
        right_xy = model(right_df[['x', 'y']].to_numpy(dtype=float))
        right_df['x_stitch'] = right_xy[:, 0]
        right_df['y_stitch'] = right_xy[:, 1]
        stitched_points = pd.concat([left_df, right_df], ignore_index=True)
        if len(stitched_points) > max_points_per_pair:
            stitched_points = stitched_points.sample(max_points_per_pair, random_state=SEED)

        dists, idx = tree.query(stitched_points[['x_stitch', 'y_stitch']].to_numpy(dtype=float), k=1)
        nearest_parent = parent_df.iloc[idx].reset_index(drop=True)
        coord_errors.extend(dists.tolist())
        within_tol.extend((dists <= nearest_tolerance).astype(float).tolist())
        misorientation_values.append(point_orientation_misorientation_deg(stitched_points.reset_index(drop=True), nearest_parent))
        evaluated_pairs += 1

    rmse = float(np.sqrt(np.mean(np.square(coord_errors)))) if coord_errors else np.nan
    return {
        'parent_file2_ang': str(parent_ang_path),
        'evaluated_tile_pairs': int(evaluated_pairs),
        'skipped_tile_pairs': int(skipped_pairs),
        'parent_coordinate_rmse': finite_float(rmse),
        'parent_mean_nearest_coordinate_error': finite_float(np.mean(coord_errors)) if coord_errors else None,
        'parent_match_fraction_within_tolerance': finite_float(np.mean(within_tol)) if within_tol else None,
        'parent_mean_misorientation_deg': finite_float(np.nanmean(misorientation_values)) if misorientation_values else None,
        'nearest_tolerance': nearest_tolerance,
    }


if RUN_PARENT_FILE2_VALIDATION:
    assert not FILE2_USED_FOR_TUNING, 'FILE2_FOLDER was used for tuning; parent validation must remain external.'
    assert Path(PARENT_FILE2_ANG).exists(), f'Parent File 2 ANG not found: {PARENT_FILE2_ANG}'
    folder2_decisions_for_parent = get_folder2_predictions_for_parent_validation()
    folder2_items_for_parent = get_folder2_items_for_parent_validation()
    parent_file2_metrics = validate_file2_stitch_against_parent(
        PARENT_FILE2_ANG,
        folder2_items_for_parent,
        folder2_decisions_for_parent,
    )
    (outputs_dir / 'parent_file2_validation_metrics.json').write_text(json_lib.dumps(parent_file2_metrics, indent=2))
    print('Parent File 2 validation metrics:')
    print(json_lib.dumps(parent_file2_metrics, indent=2))
else:
    parent_file2_metrics = {}
    print('Parent Big ANG File 2 validation is defined but not run. Set RUN_PARENT_FILE2_VALIDATION = True after paths are correct.')


## 15. Final Summary Cell

This cell prints the stitching demo transform, validation metrics, and the complete GATv2 evaluation summary when available.


In [ ]:
print('Estimated affine transform, B observed -> A frame:')
print(model_b_to_a.params)
print('\nValidation metrics:')
for k, v in metrics.items():
    print(f'{k}: {v}')

print('\nNext steps:')
print('- Crystal-symmetry-aware misorientation is enabled; update PHASE_SYMMETRY for non-cubic phases.')
print('- Replace approximate connected-component segmentation with validated grain IDs when available.')
print('- Use overlap-specific subgraphs and stronger ODF/pole-figure descriptors for difficult textures.')
print('- Continue expanding GATv2 training, per-phase/per-texture validation, and held-out parent-map checks.')

if 'eval_summary' in globals() and eval_summary:
    print('\nComplete GATv2 evaluation summary loaded from metrics.json:')
    print('best_validation_loss:', eval_summary['best_validation_loss'])
    print('test_accuracy:', eval_summary['classification']['accuracy'])
    print('f1:', eval_summary['classification']['f1'])
    print('roc_auc:', eval_summary['classification']['roc_auc'])
    print('pr_auc:', eval_summary['classification']['pr_auc'])
    print('rmse_stitching_error:', eval_summary['stitching']['rmse_stitching_error'])
    print('mean_misorientation_error:', eval_summary['stitching']['mean_misorientation_error'])
    print('grain_continuity_score:', eval_summary['stitching']['grain_continuity_score'])

if 'parent_file2_metrics' in globals() and parent_file2_metrics:
    print('\nParent Big ANG File 2 validation summary:')
    print('parent_coordinate_rmse:', parent_file2_metrics['parent_coordinate_rmse'])
    print('parent_mean_misorientation_deg:', parent_file2_metrics['parent_mean_misorientation_deg'])
    print('parent_match_fraction_within_tolerance:', parent_file2_metrics['parent_match_fraction_within_tolerance'])

